# Synthetic Insect-on-Tray Image Generator

This notebook segments insects from BIOSCAN petri-dish images using **SAM-2** and composites them onto tray backgrounds with physically correct sizing and realistic blending.

### Pipeline Overview
1. **Configuration** — GPU selection, paths, physical dimensions
2. **Load metadata** — Parse `selected_classes_original.csv` for `area_fraction` and `scale_factor`
3. **SAM-2 segmentation** — Automatically segment insects from BIOSCAN images with multi-point prompting
4. **Size calibration** — Map insect real-world size (from `scale_factor` × petri dish diameter) to tray pixel space
5. **Composition & blending** — Place insects on trays with Poisson blending for authentic appearance
6. **Batch generation** — Produce synthetic tray images at scale


---
## Cell 1 — GPU Selection & Environment Setup


In [ ]:
# ============================================================
# GPU SELECTION — Change this to pick your GPU
# ============================================================
GPU_ID = 3  # <-- Set your GPU number here (0, 1, 2, ...)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)

import torch
print(f"Using GPU {GPU_ID}: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
# ============================================================
# Install dependencies (run once)
# ============================================================
# Primary segmenter — BiRefNet (from HuggingFace):
#   !pip install transformers timm einops kornia
#
# Utilities:
#   !pip install opencv-python-headless pandas numpy matplotlib Pillow tqdm
#
# BiRefNet weights download automatically from HuggingFace on first run
# (model: ZhengPeng7/BiRefNet, ~890 MB). Requires internet on first launch.
#
# SAM-2 is no longer used in the segmentation pipeline (v10). The
# SAM2_CHECKPOINT/SAM2_CONFIG constants in the next cell are kept as
# vestiges and can be ignored.


---
## Cell 2 — Configuration & Constants


In [ ]:
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random
import gc
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS — Adjust these to your directory structure
# ============================================================
BASE_DIR = Path(".")  # Root of your project
TRAY_DIR = BASE_DIR / "3DTRAP_Dirt_Dataset"
BIOSCAN_DIR = BASE_DIR / "bioscan_balanced_images_20000_original"
CSV_PATH = BASE_DIR / "selected_classes_original_20000.csv"
# WHERE THE TRAYS ARE WRITTEN — all in home, next to everything else.
OUTPUT_DIR = BASE_DIR / "DATASET_20000_mix"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The cutouts already sit inside that folder. They are only READ, so sharing a
# filesystem with the output costs nothing.
CUTOUT_DIR = OUTPUT_DIR / "cutouts"

# TRAY SCALE — the setting that decides how well this transfers to real data.
#
# Insect size is not copied from the cutout; cell 13 derives it physically:
#     px_per_mm = tray_width_px / TRAY_WIDTH_MM        (320 mm)
#     insect_px = scale_factor x 8 mm x px_per_mm
# so the tray's pixel width IS the insect's pixel size.
#
#   3DTRAP_Dirt_Dataset backgrounds  2644 px ->  8.26 px/mm -> a 9 mm beetle at 74 px
#   real annotated 3DTRAB trays     ~3473 px -> 10.85 px/mm -> the same beetle at 98 px
#
# Left native, every synthetic insect is 24% smaller than the ones the model has to
# detect. Resizing the background to 3473 closes that. The BACKGROUND gains no real
# detail — it is interpolated — but each insect is then rendered at its true
# deployment size from the full-resolution BIOSCAN cutout instead of being squeezed
# to 76% of it, and the insect is what is being detected.
TRAY_TARGET_WIDTH = 3473          # None leaves backgrounds at their native width

# SPACE. At 3473 px a tray is 1.7x the pixels, so PNG would need ~805 GB against the
# ~716 GB free. JPEG q95 needs ~316 GB. The COCO export re-encodes to q90 either way,
# so "jpg" means the tray is compressed twice rather than once — measured at 0.07 dB
# PSNR in the final exported image (34.22 -> 34.15), which the q90 export dominates.
TRAY_FORMAT = "jpg"
assert TRAY_FORMAT in ("png", "jpg")
TRAY_EXT     = "." + TRAY_FORMAT
_TRAY_PARAMS = [cv2.IMWRITE_JPEG_QUALITY, 95] if TRAY_FORMAT == "jpg" else []

# Every path here is relative to BASE_DIR, which is Path(".") — i.e. whatever
# directory the kernel was started in, NOT necessarily ~/MA_Atul. Printed resolved so
# there is no guessing about where things will actually be read from and written to.
print(f"Working dir:   {Path('.').resolve()}")
print(f"Output dir:    {OUTPUT_DIR.resolve()}   (trays as {TRAY_FORMAT})")
if TRAY_TARGET_WIDTH:
    print(f"Tray scale:    backgrounds -> {TRAY_TARGET_WIDTH} px wide = "
          f"{TRAY_TARGET_WIDTH / 320.0:.2f} px/mm   (real 3DTRAB is 10.85)")
else:
    print("Tray scale:    backgrounds left native")
if (CUTOUT_DIR / "cutouts_metadata.csv").exists():
    # This folder has accumulated cutouts from several runs, so the file count on disk
    # says nothing by itself. What matters is how many of the CSV's rows resolve to a
    # file HERE — that is the pool the generator actually draws from.
    _cm = pd.read_csv(CUTOUT_DIR / "cutouts_metadata.csv", usecols=["processid"])
    _on_disk = {f.stem for f in CUTOUT_DIR.glob("*.png")}
    _listed  = set(_cm["processid"].astype(str))
    _usable  = _listed & _on_disk
    print(f"Cutouts:       {CUTOUT_DIR.resolve()}")
    print(f"               {len(_on_disk):,} .png on disk | {len(_listed):,} rows in the "
          f"CSV | {len(_usable):,} usable")
    if len(_on_disk) > len(_listed) * 1.2:
        print(f"               ({len(_on_disk) - len(_listed):,} extra .png are not in "
              f"this CSV — leftovers from earlier runs, ignored)")
else:
    print(f"Cutouts:       NO cutouts_metadata.csv at {CUTOUT_DIR.resolve()}")

# SAM-2 model checkpoint path
SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

# ============================================================
# PHYSICAL DIMENSIONS (in mm)
# ============================================================
TRAY_WIDTH_MM = 320.0    # 32 cm
TRAY_HEIGHT_MM = 260.0   # 26 cm
PETRI_DISH_DIAMETER_MM = 8.0  # 8 mm petri dish

# ============================================================
# GENERATION SETTINGS
# ============================================================
INSECTS_PER_TRAY = (30, 50)     # Min/max insects per generated tray
PLACEMENT_MARGIN = 0.05          # Keep insects 5% away from tray edges
ROTATION_RANGE = (0, 360)        # Random rotation in degrees
NUM_TRAYS_TO_GENERATE = 20000     # Clean trays to produce (brown adds +35% -> ~27000 total)

# Seed for reproducibility (set to None for random)
RANDOM_SEED = 42
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

print(f"Tray images:   {sum(1 for p in TRAY_DIR.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg'))} found (recursive)")
print(f"BIOSCAN images: {len(list(BIOSCAN_DIR.glob('*')))} found")


---
## Cell 3 — Load & Parse Metadata CSV


In [ ]:
# Load the BIOSCAN metadata
df = pd.read_csv(CSV_PATH)
print(f"Total entries in CSV: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nKey columns for sizing:")
print(df[['processid', 'image_measurement_value', 'area_fraction', 'scale_factor']].describe())

# Build a lookup: processid -> metadata
# The processid matches the BIOSCAN image filename (without extension)
metadata_lookup = df.set_index('processid')[['area_fraction', 'scale_factor', 'image_measurement_value', 'image_path']].to_dict('index')

# Show the entries that match our example images
example_ids = [p.stem for p in BIOSCAN_DIR.glob('*')]
print(f"\nExample BIOSCAN images and their metadata:")
for eid in example_ids:
    if eid in metadata_lookup:
        m = metadata_lookup[eid]
        print(f"  {eid}: area_fraction={m['area_fraction']:.4f}, scale_factor={m['scale_factor']:.4f}, "
              f"measurement={m['image_measurement_value']:.0f}")
    else:
        print(f"  {eid}: NOT FOUND in CSV")


---
## Cell 4 — Initialize SAM-2 Model


In [ ]:
# ============================================================
# BiRefNet (primary segmenter) + SAM-2 placeholders
# ============================================================
# BiRefNet is a Dichotomous Image Segmentation model — purpose-built for
# fine structures (insect legs, wings, hair). It's our main segmenter.
# SAM-2 references are kept as None so downstream cells that pass
# `predictor, mask_generator` positionally still work.
#
# NEW DEPENDENCIES (run once):
#   !pip install transformers timm einops kornia
#
from transformers import AutoModelForImageSegmentation
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading BiRefNet on device: {device}")

# Model choices:
#   'ZhengPeng7/BiRefNet'      -> general DIS, 1024x1024 input  (default)
#   'ZhengPeng7/BiRefNet_HR'   -> 2048x2048 input, best for fine structures
#   'ZhengPeng7/BiRefNet_lite' -> smaller, faster
BIREFNET_MODEL_ID  = 'ZhengPeng7/BiRefNet_HR'
BIREFNET_INPUT_SIZE = 2048

birefnet = AutoModelForImageSegmentation.from_pretrained(
    BIREFNET_MODEL_ID, trust_remote_code=True
)
birefnet.to(device)
birefnet.eval()
torch.set_float32_matmul_precision('high')

birefnet_transform = transforms.Compose([
    transforms.Resize((BIREFNET_INPUT_SIZE, BIREFNET_INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# SAM-2 stubs — kept so old call sites don't break.
predictor = None
mask_generator = None

print(f"BiRefNet loaded: {BIREFNET_MODEL_ID} @ {BIREFNET_INPUT_SIZE}x{BIREFNET_INPUT_SIZE}")


---
## Cell 5 — Two-Step Segmentation (v8 — Metadata-Guided)

**Step 1**: SAM-2 auto mask generator → find all objects → filter for insect candidates
- Area filter raised to **80%** of dish (large insects are valid)
- Merge overlapping/adjacent masks (captures fragmented legs)
- **NEW**: Uses `image_measurement_value` (exact target pixel count) to rank clusters — clusters whose total area is close to the target get highest score
- **NEW**: Uses `area_fraction` and `scale_factor` as soft priors for bbox/size sanity checks

**Quality gate**: If Step 1 mask is good AND area matches target, **use it directly** (skip Step 2)

**Step 2** (only if needed): SAM-2 image predictor with:
- Box from Step 1 + **positive points from Step 1 mask** (prevents blob expansion)
- **NEW**: Picks the multimask output that best matches `image_measurement_value`, not just argmax confidence
- **NEW**: Rejects Step 2 output if area deviates >5× from target (tighter than old 3× blob-only guard)
- Falls back to Step 1 if Step 2 drifts further from target than Step 1 was


In [ ]:
# ============================================================
# SEGMENTATION TOOLKIT (v10 — BiRefNet + dish subtraction)
# ============================================================

# ---------- petri dish detection (unchanged, still useful) ----------
def detect_petri_dish(image_bgr):
    """Robust petri dish detection — hybrid Hough + colour approach."""
    h, w = image_bgr.shape[:2]
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    img_cx, img_cy = w // 2, h // 2
    ideal_r = int(min(h, w) * 0.30)
    min_r, max_r = int(ideal_r * 0.70), int(ideal_r * 1.20)

    all_circles = []
    for blur_k in [7, 9, 13]:
        gb = cv2.GaussianBlur(gray, (blur_k, blur_k), 2)
        for dp in [1.0, 1.2, 1.5]:
            for p1 in [30, 50, 80]:
                for p2 in [20, 30, 50]:
                    res = cv2.HoughCircles(
                        gb, cv2.HOUGH_GRADIENT, dp=dp,
                        minDist=min(h, w) // 3,
                        param1=p1, param2=p2,
                        minRadius=min_r, maxRadius=max_r)
                    if res is not None:
                        for c in res[0]:
                            all_circles.append((int(round(c[0])), int(round(c[1])), int(round(c[2]))))
    if all_circles:
        def score(c):
            cx, cy, r = c
            d = np.sqrt((cx - img_cx)**2 + (cy - img_cy)**2)
            ds = max(0, 1 - d / (min(h, w) * 0.12))
            rs = max(0, 1 - abs(r - ideal_r) / (ideal_r * 0.30))
            return ds * 0.55 + rs * 0.45
        scored = sorted(all_circles, key=score, reverse=True)
        if score(scored[0]) > 0.5:
            return tuple(scored[0])

    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    b_ch = lab[:, :, 2].astype(np.float32)
    l_ch = lab[:, :, 0].astype(np.float32)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    cw = np.exp(-((xx - img_cx)**2 + (yy - img_cy)**2) / (2 * (ideal_r * 1.3)**2))
    dish_map = (b_ch / 255) * 0.5 + (l_ch / 255) * 0.3 + cw * 0.2
    thr = np.percentile(dish_map, 60)
    binary = (dish_map > thr).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    M = cv2.moments(binary)
    if M['m00'] > 0:
        dish_cx = int(M['m10'] / M['m00'])
        dish_cy = int(M['m01'] / M['m00'])
        if abs(dish_cx - img_cx) < w * 0.15 and abs(dish_cy - img_cy) < h * 0.15:
            return (dish_cx, dish_cy, ideal_r)
    return (img_cx, img_cy, ideal_r)


# ---------- BiRefNet inference ----------
def birefnet_segment(image_rgb):
    """Run BiRefNet — returns a float [0,1] saliency map at full resolution.
    Matches the input tensor's dtype to the model's weight dtype so that
    fp16-loaded models (common in shared envs like rembg_gpu_env) work
    transparently alongside fp32-loaded ones."""
    h, w = image_rgb.shape[:2]
    pil = Image.fromarray(image_rgb)
    x = birefnet_transform(pil).unsqueeze(0).to(device)
    # Match the model's dtype (handles both fp16 and fp32 weights uniformly)
    model_dtype = next(birefnet.parameters()).dtype
    x = x.to(model_dtype)
    with torch.no_grad():
        preds = birefnet(x)[-1].sigmoid().float().cpu()
    m_small = preds[0].squeeze().numpy().astype(np.float32)
    return cv2.resize(m_small, (w, h), interpolation=cv2.INTER_LINEAR)


# ---------- dish subtraction (classical fallback) ----------
def dish_subtraction_mask(image_rgb, cx, cy, dish_radius,
                         sample_r_inner=0.55, sample_r_outer=0.85,
                         use_r=0.92):
    """
    Sample the dish background colour from an annulus well outside the centre,
    then flag every in-dish pixel whose LAB distance from that background
    exceeds an adaptive threshold. Returns a bool mask or None.
    """
    h, w = image_rgb.shape[:2]
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, A, B = lab[:, :, 0], lab[:, :, 1], lab[:, :, 2]

    yy, xx = np.mgrid[0:h, 0:w]
    d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)

    in_dish     = d <= (dish_radius * use_r)
    sample_ring = (d >= dish_radius * sample_r_inner) & (d <= dish_radius * sample_r_outer)
    if sample_ring.sum() < 200:
        return None

    bg_L = np.median(L[sample_ring])
    bg_A = np.median(A[sample_ring])
    bg_B = np.median(B[sample_ring])
    dE = np.sqrt((L - bg_L) ** 2 + (A - bg_A) ** 2 + (B - bg_B) ** 2)

    thr = max(np.percentile(dE[in_dish], 82), 6.0)
    candidate = ((dE > thr) & in_dish).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    candidate = cv2.morphologyEx(candidate, cv2.MORPH_OPEN, k, iterations=1)
    candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, k, iterations=2)

    n, lbl, stats, centroids = cv2.connectedComponentsWithStats(candidate)
    if n <= 1:
        return None
    best, best_score = None, -1
    for i in range(1, n):
        ccx, ccy = centroids[i]
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 30:
            continue
        dist = np.sqrt((ccx - cx) ** 2 + (ccy - cy) ** 2) / (dish_radius + 1)
        s = area * max(0.1, 1.0 - dist)
        if s > best_score:
            best_score, best = s, i
    if best is None:
        return None
    return (lbl == best)


# ---------- mask post-processing ----------
def clip_to_dish(mask, cx, cy, dish_radius, pad=0.95):
    h, w = mask.shape
    yy, xx = np.mgrid[0:h, 0:w]
    d2 = (xx - cx) ** 2 + (yy - cy) ** 2
    return mask & (d2 <= (dish_radius * pad) ** 2)


def keep_central_blob(mask, cx, cy, min_frac=0.05):
    """Keep the connected component closest to dish centre; drop fragments."""
    result = mask.astype(np.uint8)
    n, lbl, stats, centroids = cv2.connectedComponentsWithStats(result)
    if n <= 1:
        return mask.astype(bool)
    best, best_score = 0, -1
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 30:
            continue
        ccx, ccy = centroids[i]
        dist = np.sqrt((ccx - cx) ** 2 + (ccy - cy) ** 2)
        s = area / (dist + 10.0)
        if s > best_score:
            best_score, best = s, i
    if best == 0:
        return mask.astype(bool)
    # Also keep fragments that touch the central blob bbox (wings)
    x0, y0 = stats[best, cv2.CC_STAT_LEFT], stats[best, cv2.CC_STAT_TOP]
    x1 = x0 + stats[best, cv2.CC_STAT_WIDTH]
    y1 = y0 + stats[best, cv2.CC_STAT_HEIGHT]
    pad = 10
    keep = (lbl == best).astype(np.uint8)
    for i in range(1, n):
        if i == best:
            continue
        area = stats[i, cv2.CC_STAT_AREA]
        if area < max(30, stats[best, cv2.CC_STAT_AREA] * min_frac):
            continue
        ix0 = stats[i, cv2.CC_STAT_LEFT]; iy0 = stats[i, cv2.CC_STAT_TOP]
        ix1 = ix0 + stats[i, cv2.CC_STAT_WIDTH]; iy1 = iy0 + stats[i, cv2.CC_STAT_HEIGHT]
        if ix1 < x0 - pad or ix0 > x1 + pad or iy1 < y0 - pad or iy0 > y1 + pad:
            continue
        keep[lbl == i] = 1
    return keep.astype(bool)


def postprocess_mask(mask, cx, cy, dish_radius):
    """Keep central blob + nearby fragments, gentle morph close.
    Intentionally does NOT hard-clip to the dish circle — the Hough detector
    can be slightly off, and clipping truncates wings/legs that legitimately
    extend past the detected radius. keep_central_blob drops far-away
    fragments (scale bars, dish-rim artifacts) via a distance/area heuristic,
    which is gentler than a pixel-hard circular crop."""
    m = mask.astype(bool)
    m = keep_central_blob(m, cx, cy)
    m_u8 = m.astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    m_u8 = cv2.morphologyEx(m_u8, cv2.MORPH_CLOSE, k, iterations=1)
    return m_u8.astype(bool)


def extract_insect_rgba(image_bgr, mask):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    alpha = mask.astype(np.float32) * 255
    alpha = cv2.GaussianBlur(alpha, (5, 5), 1.0)
    alpha = np.clip(alpha, 0, 255).astype(np.uint8)
    rgba = np.dstack([image_rgb, alpha])
    rows = np.any(mask, axis=1); cols = np.any(mask, axis=0)
    if not np.any(rows) or not np.any(cols):
        return None, None
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pad = 5
    rmin = max(0, rmin - pad); rmax = min(mask.shape[0] - 1, rmax + pad)
    cmin = max(0, cmin - pad); cmax = min(mask.shape[1] - 1, cmax + pad)
    return rgba[rmin:rmax + 1, cmin:cmax + 1], (rmin, rmax, cmin, cmax)


# ---------- area-matching helpers ----------
STRICT_AREA_BAND = (0.90, 1.05)
LOOSE_AREA_BAND  = (0.50, 2.00)

def _tier_of(area, target):
    if target is None or target <= 0:
        return 2  # no target => treat as acceptable-but-unknown
    r = area / target
    if STRICT_AREA_BAND[0] <= r <= STRICT_AREA_BAND[1]:
        return 1
    if LOOSE_AREA_BAND[0] <= r <= LOOSE_AREA_BAND[1]:
        return 2
    return 3

def _log_gauss_score(area, target, sigma=0.30):
    if target is None or target <= 0 or area <= 0:
        return 0.0
    return float(np.exp(-0.5 * (np.log(area / target) / sigma) ** 2))


def pick_best_threshold(prob_map, cx, cy, dish_radius, target_area,
                        thresholds=(0.20, 0.30, 0.40, 0.50, 0.60, 0.70)):
    """Sweep thresholds on a saliency map; return list of (name, mask)."""
    candidates = []
    for t in thresholds:
        m = (prob_map > t)
        m = postprocess_mask(m, cx, cy, dish_radius)
        a = int(np.sum(m))
        if a < 30:
            continue
        candidates.append((f"birefnet_thr{t:.2f}", m, a))
    return candidates


def saliency_is_diffuse(prob, frac_thresh=0.30, min_peak=0.50):
    """BiRefNet failure detector — True if the saliency has no clear object.
    Triggers when the map is nearly uniform (whole dish lit up) or has no
    confident peak at all."""
    if prob.max() < min_peak:
        return True
    bright_frac = (prob > 0.5).sum() / prob.size
    return bool(bright_frac > frac_thresh)


def birefnet_segment_cropped(image_rgb, cx, cy, dish_radius, pad_ratio=1.15):
    """Run BiRefNet on a dish-cropped region — the insect occupies a larger
    fraction of the input, so the model sees it at effectively higher
    resolution. Returns a full-resolution saliency map (zeros outside crop)
    or None if the crop is degenerate."""
    h, w = image_rgb.shape[:2]
    r = int(dish_radius * pad_ratio)
    x1 = max(0, cx - r); y1 = max(0, cy - r)
    x2 = min(w, cx + r); y2 = min(h, cy + r)
    crop = image_rgb[y1:y2, x1:x2]
    if crop.shape[0] < 32 or crop.shape[1] < 32:
        return None
    crop_prob = birefnet_segment(crop)
    full = np.zeros((h, w), dtype=np.float32)
    full[y1:y2, x1:x2] = crop_prob
    return full


def area_matched_mask(prob, target_area, cx, cy, dish_radius,
                      min_saliency=0.05, n_iter=30):
    """Binary-search the saliency threshold T so that (prob > T) has ~target_area
    pixels. Produces a clean connected region following saliency contours —
    wings stay proportional to their saliency, body at full weight. Guarantees
    a TIER1 area hit whenever BiRefNet saw the insect at all."""
    if target_area is None or target_area <= 0:
        return None
    target = int(target_area)
    lo, hi = float(prob.min()), float(prob.max())
    best_mask, best_diff = None, float("inf")
    for _ in range(n_iter):
        if hi - lo < 1e-4:
            break
        mid = 0.5 * (lo + hi)
        cnt = int((prob > mid).sum())
        diff = abs(cnt - target)
        if diff < best_diff:
            best_diff = diff
            best_mask = (prob > mid)
        if cnt > target:
            lo = mid
        else:
            hi = mid
    if best_mask is None:
        return None
    best_mask = best_mask & (prob > min_saliency)
    if best_mask.sum() < 30:
        return None
    return keep_central_blob(best_mask, cx, cy)


print("Segmentation toolkit loaded (v10 — BiRefNet + dish subtraction).")


---
## Cell 6 — Size Calibration Functions

### How sizing works:

- **BIOSCAN images**: Each 1024×768 image shows an insect on an 8 mm petri dish
- **`scale_factor`** in the CSV: Proportional to the insect's real-world body length relative to the petri dish. Specifically, `scale_factor ≈ insect_body_length / petri_dish_diameter` (approximately)
- **`area_fraction`**: The fraction of the image area occupied by the insect
- **Tray images**: 2610×2172 px representing a 32 cm × 26 cm physical tray

**Scaling pipeline:**
1. Estimate the petri dish size in pixels from the BIOSCAN image
2. Calculate `px_per_mm_bioscan = dish_diameter_px / 8 mm`
3. Calculate `px_per_mm_tray = tray_width_px / 320 mm`
4. Scale the cropped insect by `px_per_mm_tray / px_per_mm_bioscan`


In [ ]:
def calculate_insect_target_size(
    insect_crop_rgba,
    dish_radius_px,
    tray_shape,
    scale_factor,
    area_fraction
):
    """
    Calculate the target pixel size for an insect when placed on a tray.

    Parameters:
    -----------
    insect_crop_rgba : np.ndarray
        Cropped RGBA insect image from BIOSCAN
    dish_radius_px : int
        Detected petri dish radius in pixels (in the BIOSCAN image)
    tray_shape : tuple
        (height, width) of the tray image in pixels
    scale_factor : float
        From CSV — relates insect size to petri dish
    area_fraction : float
        From CSV — fraction of image area occupied by the insect

    Returns:
    --------
    target_height, target_width : int
        The pixel dimensions the insect should be when placed on the tray
    """
    tray_h, tray_w = tray_shape[:2]
    dish_diameter_px = dish_radius_px * 2

    # Pixels per mm in BIOSCAN image
    px_per_mm_bioscan = dish_diameter_px / PETRI_DISH_DIAMETER_MM

    # Pixels per mm in tray image
    px_per_mm_tray = tray_w / TRAY_WIDTH_MM

    # The base scale ratio: how to convert BIOSCAN pixels to tray pixels
    base_scale = px_per_mm_tray / px_per_mm_bioscan

    # The scale_factor from the CSV tells us about relative insect size.
    # We use it to double-check/adjust: if scale_factor is very different
    # from what the crop dimensions imply, we trust the CSV more.
    #
    # Insect real-world length ≈ scale_factor * PETRI_DISH_DIAMETER_MM
    # Insect real-world length in tray pixels = scale_factor * PETRI_DISH_DIAMETER_MM * px_per_mm_tray

    crop_h, crop_w = insect_crop_rgba.shape[:2]

    # Method 1: Direct geometric scaling
    target_w_geom = int(crop_w * base_scale)
    target_h_geom = int(crop_h * base_scale)

    # Method 2: Use scale_factor to determine the major axis length
    insect_major_axis_mm = scale_factor * PETRI_DISH_DIAMETER_MM
    insect_major_axis_tray_px = insect_major_axis_mm * px_per_mm_tray

    # Scale the crop so the major axis matches
    major_axis_crop = max(crop_h, crop_w)
    if major_axis_crop > 0:
        sf_scale = insect_major_axis_tray_px / major_axis_crop
    else:
        sf_scale = base_scale

    target_w_sf = int(crop_w * sf_scale)
    target_h_sf = int(crop_h * sf_scale)

    # Blend both methods (weighted average, trusting scale_factor more)
    alpha_blend = 0.7  # Weight for scale_factor method
    target_w = int(alpha_blend * target_w_sf + (1 - alpha_blend) * target_w_geom)
    target_h = int(alpha_blend * target_h_sf + (1 - alpha_blend) * target_h_geom)

    # Safety: ensure minimum visible size (at least 3x3 px on tray)
    target_w = max(target_w, 3)
    target_h = max(target_h, 3)

    # Safety: cap at 15% of tray dimension (no insect is that large)
    target_w = min(target_w, int(tray_w * 0.15))
    target_h = min(target_h, int(tray_h * 0.15))

    return target_h, target_w


# Quick sanity check with known values
# AACTA1465-20: scale_factor=1.125 (large insect ~9mm)
# On a 2610x2172 tray (320mm wide): px/mm = 8.15
# Expected major axis ≈ 1.125 * 8 * 8.15 ≈ 73 px on tray
print("Size calibration functions defined.")
print(f"Tray resolution: {2610/TRAY_WIDTH_MM:.2f} px/mm")
print(f"Example: scale_factor=1.125 insect → ~{1.125 * PETRI_DISH_DIAMETER_MM:.1f}mm → "
      f"~{1.125 * PETRI_DISH_DIAMETER_MM * (2610/TRAY_WIDTH_MM):.0f}px on tray")


---
## Cell 7 — Blending & Composition Functions

Two blending modes:
- **Poisson blending** (OpenCV `seamlessClone`) — produces the most realistic results by matching colour gradients
- **Alpha blending** — faster fallback with feathered edges


In [ ]:
from PIL import ImageFilter

def extend_edge_colors(img_rgba_pil, radius=2):
    """Spread opaque pixel colors into semi-transparent edge pixels.
    Prevents dark fringing when compositing — fills the feathered alpha
    border with the nearest opaque insect color instead of black."""
    arr = np.array(img_rgba_pil).astype(np.float32)
    alpha = arr[:, :, 3] / 255.0

    weight = alpha.copy()
    weight_blur = cv2.GaussianBlur(weight, (0, 0), sigmaX=radius)
    weight_blur = np.clip(weight_blur, 1e-6, None)

    result = arr.copy()
    semi = (alpha > 0) & (alpha < 1.0)
    for c in range(3):
        chan_blur = cv2.GaussianBlur(arr[:, :, c] * weight, (0, 0), sigmaX=radius)
        filled = chan_blur / weight_blur
        result[:, :, c][semi] = np.clip(filled[semi], 0, 255)

    return Image.fromarray(result.astype(np.uint8), "RGBA")


def blend_insect_edges_with_background(insect_rgba_pil, tray_rgba_pil, x, y):
    """At semi-transparent edge pixels, mix 70% insect + 30% tray colour
    so the insect blends into the local tray background."""
    # Crop the tray FIRST. Only this window is ever read, and materialising the whole
    # tray as float32 (88 MB at 2644x2108) once per insect was the single most
    # expensive line in the generator. Output is bit-identical.
    insect_arr = np.array(insect_rgba_pil).astype(np.float32)

    h, w = insect_arr.shape[:2]
    tray_w, tray_h = tray_rgba_pil.size
    if y + h > tray_h or x + w > tray_w:
        return insect_rgba_pil
    tray_arr = np.asarray(tray_rgba_pil.crop((x, y, x + w, y + h))).astype(np.float32)

    alpha = insect_arr[:, :, 3] / 255.0
    edge_mask = (alpha > 0.05) & (alpha < 0.9)

    for c in range(3):
        insect_arr[:, :, c][edge_mask] = (
            0.7 * insect_arr[:, :, c][edge_mask] +
            0.3 * tray_arr[:, :, c][edge_mask]
        )

    return Image.fromarray(insect_arr.astype(np.uint8), "RGBA")


def add_shadow_pil(tray_rgba_pil, insect_pil, x, y,
                   shadow_offset=(2, 2), shadow_blur_radius=6, shadow_opacity=0.22):
    """Add a subtle drop shadow under the insect using PIL alpha compositing.

    Only the rectangle the shadow can actually reach is composited. The previous
    version built a full-tray RGBA layer (22 MB at 2644x2108) and alpha-composited
    the whole tray for EVERY insect — about a million times over a 27,000-tray run.
    Compositing a fully transparent pixel is the identity, so restricting to the
    shadow's own box gives bit-identical output; verified pixel-exact over full
    40-insect trays.
    """
    shadow_alpha = insect_pil.split()[-1]
    shadow_alpha = shadow_alpha.point(lambda p: int(p * shadow_opacity))
    shadow_alpha = shadow_alpha.filter(ImageFilter.GaussianBlur(shadow_blur_radius))

    iw, ih = insect_pil.size
    tw, th = tray_rgba_pil.size
    px, py = x + shadow_offset[0], y + shadow_offset[1]
    L, T = max(0, px), max(0, py)
    R, B = min(tw, px + iw), min(th, py + ih)
    if R <= L or B <= T:
        return tray_rgba_pil

    shadow_layer = Image.new("RGBA", (R - L, B - T), (0, 0, 0, 0))
    shadow_layer.paste(
        Image.new("RGBA", insect_pil.size, (0, 0, 0, 255)),
        (px - L, py - T),
        shadow_alpha
    )
    # in place, then returned — both call sites do `tray = add_shadow_pil(tray, ...)`
    tray_rgba_pil.paste(
        Image.alpha_composite(tray_rgba_pil.crop((L, T, R, B)), shadow_layer), (L, T))
    return tray_rgba_pil


def place_insect_on_tray_pil(tray_rgba_pil, insect_rgba_pil, x, y):
    """
    Place a segmented insect onto the tray using PIL alpha compositing.
    Pipeline:
      1. Soften alpha edges (Gaussian blur 0.8px)
      2. Extend edge colors (prevent dark fringing)
      3. Add drop shadow underneath
      4. Blend edge pixels with local tray colour
      5. Paste with alpha mask
    """
    iw, ih = insect_rgba_pil.size
    tw, th = tray_rgba_pil.size

    # Bounds check
    if x + iw > tw or y + ih > th or x < 0 or y < 0:
        # Crop to fit
        cx1 = max(0, -x)
        cy1 = max(0, -y)
        cx2 = min(iw, tw - x)
        cy2 = min(ih, th - y)
        if cx2 <= cx1 or cy2 <= cy1:
            return tray_rgba_pil
        insect_rgba_pil = insect_rgba_pil.crop((cx1, cy1, cx2, cy2))
        x = max(0, x)
        y = max(0, y)

    # 1. Soften alpha edges
    alpha = insect_rgba_pil.split()[-1]
    alpha = alpha.filter(ImageFilter.GaussianBlur(0.8))
    insect_rgba_pil.putalpha(alpha)

    # 2. Extend edge colours to prevent dark fringing
    insect_rgba_pil = extend_edge_colors(insect_rgba_pil, radius=2)

    # 3. Shadow
    tray_rgba_pil = add_shadow_pil(tray_rgba_pil, insect_rgba_pil, x, y)

    # 4. Blend edge pixels with local tray background
    insect_rgba_pil = blend_insect_edges_with_background(
        insect_rgba_pil, tray_rgba_pil, x, y
    )

    # 5. Paste with alpha mask
    tray_rgba_pil.paste(insect_rgba_pil, (x, y), insect_rgba_pil)

    return tray_rgba_pil


print("Blending functions defined (v4 — PIL alpha compositing with edge defringing).")

# ============================================================
# CAMERA-REALISM PASS — run AFTER IC-Light, on the full tray
# ============================================================
# Reasons synthetic still looks fake even after harmonization:
#   1. Tray has sensor noise; pasted insects do not.
#   2. Tray was shot through a real lens (slight optical blur);
#      cutouts came from a different camera/scale & end up too crisp.
#   3. Dust / micro-debris on the substrate is missing under insects.
# This pass tackles (1) and (2) over the foreground regions only.
# ============================================================

def _sample_substrate_noise_sigma(tray_rgb, mask_u8, patch=192):
    """Estimate per-channel sensor noise sigma from a clean substrate patch
    (chosen far from any foreground)."""
    H, W = tray_rgb.shape[:2]
    free = (mask_u8 == 0).astype(np.uint8)
    # integral image to find largest insect-free square quickly
    ii = cv2.integral(free)
    best = (0, 0, -1)
    step = max(8, patch // 8)
    for yy in range(0, H - patch, step):
        for xx in range(0, W - patch, step):
            s = (ii[yy + patch, xx + patch]
                 - ii[yy, xx + patch]
                 - ii[yy + patch, xx]
                 + ii[yy, xx])
            if s > best[2]:
                best = (xx, yy, s)
    xx, yy, _ = best
    p = tray_rgb[yy:yy + patch, xx:xx + patch].astype(np.float32)
    # high-pass = noise estimate
    lp = cv2.GaussianBlur(p, (0, 0), sigmaX=1.5)
    hp = p - lp
    sig = hp.reshape(-1, 3).std(axis=0)
    return np.clip(sig, 0.5, 8.0)  # per-channel sigma


def add_camera_realism(tray_rgb, mask_u8,
                       edge_blur_sigma=0.7,
                       noise_scale=1.0,
                       contact_shadow_strength=0.18,
                       contact_shadow_radius=3):
    """In-place-ish realism pass on the foreground regions of `tray_rgb`.

    1. Soft optical blur on the foreground (matches lens MTF).
    2. Adds matched per-channel Gaussian sensor noise on the foreground.
    3. Subtle dark contact halo at the edge of each foreground blob.
    """
    if mask_u8 is None or mask_u8.sum() == 0:
        return tray_rgb
    H, W = tray_rgb.shape[:2]
    fg = (mask_u8 > 0).astype(np.float32)
    fg_soft = cv2.GaussianBlur(fg, (0, 0), sigmaX=1.0)[..., None]

    out = tray_rgb.astype(np.float32)

    # 1. Optical blur (only where there is foreground)
    if edge_blur_sigma > 0:
        blurred = cv2.GaussianBlur(out, (0, 0), sigmaX=edge_blur_sigma)
        out = fg_soft * blurred + (1.0 - fg_soft) * out

    # 2. Sensor-noise matching
    if noise_scale > 0:
        sig = _sample_substrate_noise_sigma(tray_rgb, mask_u8) * noise_scale
        rng = np.random.default_rng(12345)
        noise = rng.standard_normal((H, W, 3)).astype(np.float32) * sig[None, None, :]
        out = out + fg_soft * noise

    # 3. Contact shadow (narrow dark band around foreground edges)
    if contact_shadow_strength > 0:
        # outer ring = (dilated fg) - fg
        kern = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (2 * contact_shadow_radius + 1,
                                2 * contact_shadow_radius + 1))
        dil = cv2.dilate((mask_u8 > 0).astype(np.uint8), kern)
        ring = (dil - (mask_u8 > 0).astype(np.uint8)).astype(np.float32)
        ring = cv2.GaussianBlur(ring, (0, 0), sigmaX=contact_shadow_radius)
        ring = ring[..., None] * contact_shadow_strength
        out = out * (1.0 - ring)

    return np.clip(out, 0, 255).astype(np.uint8)


def match_foreground_brightness(tray_rgb, mask_u8,
                                strength=1.0, gain_clip=(0.25, 1.06),
                                extra_darken=0.93, sigma_frac=0.06,
                                clip_pct=(20, 80),
                                chroma_boost=1.15,
                                cast_strength=0.45,
                                dim_pivot=190.0,
                                dim_floor=0.55):
    """LAB per-pixel L* pull + chroma boost + white-balance cast transfer.

    1. L*: each fg pixel pulled toward LOCAL substrate L (gain_clip), with
       global adaptive dimming on dim trays.
    2. a*/b*: chroma_boost keeps insect colour from washing out, then a
       cast-transfer shifts the fg mean a*/b* toward the substrate mean
       a*/b* (real subjects share the ambient white balance with the bg).
    """
    if mask_u8 is None or mask_u8.sum() == 0:
        return tray_rgb
    H, W = tray_rgb.shape[:2]
    lab = cv2.cvtColor(tray_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0]; a = lab[..., 1]; b = lab[..., 2]
    fg = (mask_u8 > 0).astype(np.float32)
    bg = 1.0 - fg
    if bg.sum() < 50:
        return tray_rgb

    bg_mask = bg > 0.5
    fg_mask = fg > 0.5
    bg_L_vals = L[bg_mask]
    blo, bhi = np.percentile(bg_L_vals, clip_pct)
    L_clip = np.clip(L, blo, bhi)
    sub_global = float(np.clip(bg_L_vals, blo, bhi).mean())

    sigma = max(20.0, sigma_frac * min(H, W))
    L_smoothed  = cv2.GaussianBlur(L_clip * bg, (0, 0), sigmaX=sigma)
    bg_smoothed = cv2.GaussianBlur(bg,          (0, 0), sigmaX=sigma)
    L_local = L_smoothed / np.clip(bg_smoothed, 1e-3, None)

    eps = 1e-3
    gain = (L_local + eps) / (L + eps)
    gain = np.clip(gain, gain_clip[0], gain_clip[1])
    gain = 1.0 + strength * (gain - 1.0)
    adaptive = float(np.clip(sub_global / float(dim_pivot), float(dim_floor), 1.0))
    gain = gain * float(extra_darken) * adaptive

    fg_soft  = cv2.GaussianBlur(fg, (0, 0), sigmaX=1.5)
    fg_soft3 = fg_soft

    L_blend = (1.0 - fg_soft3) + fg_soft3 * gain
    L_new = L * L_blend

    # ── chroma: boost + saturation-weighted cast transfer ──
    # Per-pixel chroma magnitude (how saturated each pixel is, 0..~70)
    chroma_mag = np.sqrt((a - 128.0) ** 2 + (b - 128.0) ** 2)
    # Neutral pixels (low chroma) pick up MORE ambient cast,
    # already-saturated pixels keep their identity.
    cast_weight = np.clip(1.0 - chroma_mag / 25.0, 0.0, 1.0)

    if fg_mask.sum() > 50:
        a_sub = float(np.mean(a[bg_mask]))
        b_sub = float(np.mean(b[bg_mask]))
        a_fg_mean = float(np.mean(a[fg_mask]))
        b_fg_mean = float(np.mean(b[fg_mask]))
        da_global = (a_sub - a_fg_mean) * float(cast_strength)
        db_global = (b_sub - b_fg_mean) * float(cast_strength)
    else:
        da_global = db_global = 0.0

    # Boost saturation on the foreground so the chroma pull does not wash out
    if abs(chroma_boost - 1.0) > 1e-3:
        boost = (1.0 - fg_soft3) * 1.0 + fg_soft3 * float(chroma_boost)
        a_chroma = (a - 128.0) * boost + 128.0
        b_chroma = (b - 128.0) * boost + 128.0
    else:
        a_chroma, b_chroma = a, b

    # Apply cast only where (a) it is foreground and (b) pixel is desaturated
    a_new = a_chroma + fg_soft3 * cast_weight * da_global
    b_new = b_chroma + fg_soft3 * cast_weight * db_global

    lab[..., 0] = np.clip(L_new,  0, 255)
    lab[..., 1] = np.clip(a_new,  0, 255)
    lab[..., 2] = np.clip(b_new,  0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)

print("Blending + camera-realism functions defined (v7).")


---
## Cell 8 — Full Segmentation Pipeline (v8 — Metadata-Guided)

Pipeline flow:
1. Detect petri dish
2. Extract `image_measurement_value` (target area), `area_fraction`, `scale_factor` from metadata
3. **Step 1** — auto mask generator + colour-contrast filter + clustering + metadata-guided ranking
4. **Quality gate** — skip Step 2 if Step 1 mask is compact AND area is within 0.5×–2× of target
5. **Step 2** (if needed) — SAM-2 point-prompted refinement, pick multimask variant best matching target area
6. **Validation** — post-hoc check that final mask area is within 0.2×–5× of target; warn otherwise


In [ ]:
def process_bioscan_image(image_path, predictor=None, mask_generator=None,
                          metadata=None, debug=False):
    """
    v15 pipeline — shape-based rim-crescent detector.

    Rejection gate (the only reason to fall back to dish_sub):
      * saliency is diffuse (already handled by saliency_is_diffuse), OR
      * mask is a rim-crescent artifact:
          r_mean > 0.85 * dish_radius AND r_std < 0.08 * dish_radius
        i.e. pixels cluster in a thin shell at the rim.
        Real insects — even those sitting right at or outside the dish
        edge — have wider radial spread (body + legs/wings extend inward
        or outward), so they pass.
    """
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_bgr.shape[:2]
    processid = image_path.stem

    cx, cy, dish_radius = detect_petri_dish(image_bgr)
    dish_area = 3.14159 * (dish_radius ** 2)

    def _clean(v):
        try:
            if v is None: return None
            fv = float(v)
            if fv != fv or fv <= 0: return None
            return fv
        except (TypeError, ValueError):
            return None

    target_area   = _clean(metadata.get('image_measurement_value')) if metadata else None
    area_fraction = _clean(metadata.get('area_fraction'))          if metadata else None
    scale_factor  = _clean(metadata.get('scale_factor'))           if metadata else None

    # --- Metadata sanity (for SUSPECT-TARGET quality tag only) ---
    target_is_suspect = False
    if target_area is not None:
        if target_area < 300:
            target_is_suspect = True
        elif target_area > dish_area * 1.5:
            target_is_suspect = True
        elif dish_area > 0 and target_area / dish_area < 0.001:
            target_is_suspect = True

    def _radial_stats(mask):
        """Mean and std of pixel distance from dish centre, in units of dish_radius."""
        ys, xs = np.where(mask)
        if len(xs) < 10:
            return None, None
        d = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)
        return float(np.mean(d)) / max(dish_radius, 1), float(np.std(d)) / max(dish_radius, 1)

    def _is_rim_crescent(mask):
        """True iff mask looks like an optical ring artifact — thin shell on the rim."""
        r_mean, r_std = _radial_stats(mask)
        if r_mean is None:
            return False
        return (r_mean > 0.85) and (r_std < 0.08)

    def _sanity_ok(mask):
        if int(np.sum(mask)) < 30:
            return False
        return not _is_rim_crescent(mask)

    # ========================================================================
    # STEP 1 — BiRefNet primary
    # ========================================================================
    BIREFNET_THRESHOLD = 0.30
    prob = birefnet_segment(image_rgb)
    saliency_source = "birefnet_full"
    birefnet_ok = not saliency_is_diffuse(prob)
    if not birefnet_ok:
        prob_cropped = birefnet_segment_cropped(image_rgb, cx, cy, dish_radius)
        if prob_cropped is not None and not saliency_is_diffuse(prob_cropped):
            prob = prob_cropped
            saliency_source = "birefnet_crop"
            birefnet_ok = True

    bf_mask = None
    if birefnet_ok:
        bf_raw = postprocess_mask(prob > BIREFNET_THRESHOLD, cx, cy, dish_radius)
        if bf_raw.sum() >= 30:
            bf_mask = bf_raw

    # ========================================================================
    # STEP 2 — select
    # ========================================================================
    best_mask, best_name, used_fallback = None, None, False
    bf_reject_reason = None

    if bf_mask is not None and _sanity_ok(bf_mask):
        best_mask = bf_mask
        best_name = f"birefnet_thr{BIREFNET_THRESHOLD:.2f}|{saliency_source}"
    else:
        if bf_mask is None:
            bf_reject_reason = "saliency_diffuse_or_empty"
        else:
            r_mean, r_std = _radial_stats(bf_mask)
            bf_reject_reason = f"rim_crescent(r_mean={r_mean:.2f},r_std={r_std:.3f})"

        # Dish-subtraction fallback (no extra tight clipping)
        ds_raw = dish_subtraction_mask(image_rgb, cx, cy, dish_radius)
        if ds_raw is not None:
            ds_clean = postprocess_mask(ds_raw, cx, cy, dish_radius)
            if _sanity_ok(ds_clean):
                best_mask = ds_clean
                best_name = "dish_sub|fallback"
                used_fallback = True

        # If both rejected, keep whichever we have (BiRefNet preferred)
        if best_mask is None:
            if bf_mask is not None:
                best_mask = bf_mask
                best_name = f"birefnet_thr{BIREFNET_THRESHOLD:.2f}|{saliency_source}|UNVERIFIED"
            elif ds_raw is not None:
                best_mask = postprocess_mask(ds_raw, cx, cy, dish_radius)
                best_name = "dish_sub|fallback|UNVERIFIED"
                used_fallback = True

    if best_mask is None:
        print(f"  WARNING: No mask at all for {processid} (BiRefNet+dish_sub both empty)")
        return None

    clean_mask = best_mask.astype(bool)
    final_area = int(np.sum(clean_mask))

    # --- Quality tagging ---
    if target_area is not None:
        final_ratio = final_area / target_area
        if target_is_suspect:
            final_band = "SUSPECT-TARGET"
            final_tier = None
        elif STRICT_AREA_BAND[0] <= final_ratio <= STRICT_AREA_BAND[1]:
            final_band = "STRICT"
            final_tier = _tier_of(final_area, target_area)
        elif LOOSE_AREA_BAND[0] <= final_ratio <= LOOSE_AREA_BAND[1]:
            final_band = "LOOSE-OK"
            final_tier = _tier_of(final_area, target_area)
        else:
            final_band = "LOOSE-MISS"
            final_tier = _tier_of(final_area, target_area)
            if final_ratio < 0.3 or final_ratio > 4.0:
                print(f"  WARNING [LOOSE-MISS]: {processid} mask {final_area}px is "
                      f"{final_ratio:.2f}x target {int(target_area)}px "
                      f"(source: {best_name}, bf_reject: {bf_reject_reason})")
    else:
        final_ratio = None
        final_band  = "NO-TARGET"
        final_tier  = None

    best_score = _log_gauss_score(final_area, target_area) if target_area else 0.0
    tgt_str = f"tgt={int(target_area)}" if target_area else "tgt=?"
    strategy = f"{best_name}[{final_band}] score={best_score:.2f} area={final_area} {tgt_str}"

    rgba, crop_bbox = extract_insect_rgba(image_bgr, clean_mask)
    if rgba is None:
        return None

    result = {
        'rgba':          rgba,
        'mask':          clean_mask,
        'dish_radius':   dish_radius,
        'score':         best_score,
        'strategy':      strategy,
        'band':          final_band,
        'tier':          final_tier,
        'area_ratio':    final_ratio,
        'area_fraction': area_fraction if area_fraction is not None else 0.05,
        'scale_factor':  scale_factor  if scale_factor  is not None else 0.5,
        'target_area':   int(target_area) if target_area is not None else None,
        'final_area':    final_area,
        'processid':     processid,
        'image_path':    str(image_path),
        'used_fallback': used_fallback,
        'bf_reject_reason': bf_reject_reason,
    }

    if debug:
        fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
        axes[0].imshow(image_rgb)
        axes[0].add_patch(plt.Circle((cx, cy), dish_radius, fill=False,
                                     color='cyan', lw=1.5, alpha=0.5))
        tgt_txt = f"\ntarget={int(target_area)}px" if target_area else ""
        axes[0].set_title(f"Original ({processid}){tgt_txt}")

        axes[1].imshow(prob, cmap='magma', vmin=0, vmax=1)
        axes[1].set_title(f"BiRefNet saliency\n({saliency_source}, ok={birefnet_ok})")

        axes[2].imshow(clean_mask, cmap='gray')
        area_info = f"\n{final_area}px"
        if target_area:
            area_info += f" ({100 * final_area / target_area:.0f}% of target)"
        if used_fallback:
            src_tag = "FALLBACK: dish_sub"
        else:
            src_tag = f"BiRefNet thr={BIREFNET_THRESHOLD}"
        if bf_reject_reason:
            src_tag += f"\n(bf_rejected: {bf_reject_reason})"
        axes[2].set_title(f"{src_tag}{area_info}")

        axes[3].imshow(rgba)
        axes[3].set_title(f"RGBA {rgba.shape[1]}x{rgba.shape[0]}\n[{final_band}]")

        for ax in axes:
            ax.axis('off')
        plt.tight_layout(); plt.show()

    return result


print("Pipeline defined (v15 — rim-crescent shape gate, insect can be anywhere).")


---
## Cell 9 — Test Segmentation on Example Images


In [ ]:
from tqdm.auto import tqdm

MAX_DEBUG_IMAGES = 25
CUTOUT_DIR = OUTPUT_DIR / "cutouts"
CUTOUT_DIR.mkdir(exist_ok=True)

segmented_meta = []

image_files = sorted([p for p in BIOSCAN_DIR.glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
debug_files = image_files[:MAX_DEBUG_IMAGES]

for img_path in tqdm(debug_files, desc="Segmenting (debug)"):
    processid = img_path.stem
    meta = metadata_lookup.get(processid, None)

    result = process_bioscan_image(img_path, predictor, mask_generator, metadata=meta, debug=True)

    if result is not None:
        cutout_path = CUTOUT_DIR / f"{processid}.png"
        rgba_bgra = cv2.cvtColor(result['rgba'], cv2.COLOR_RGBA2BGRA)
        cv2.imwrite(str(cutout_path), rgba_bgra)

        segmented_meta.append({
            'processid': processid,
            'cutout_path': str(cutout_path),
            'dish_radius': result['dish_radius'],
            'score': result['score'],
            'strategy': result['strategy'],
            'band':       result.get('band'),
            'tier':       result.get('tier'),
            'area_ratio': result.get('area_ratio'),
            'area_fraction': result['area_fraction'],
            'scale_factor': result['scale_factor'],
            'target_area': result.get('target_area'),
            'final_area':  result.get('final_area'),
        })

        del result

print(f"\nSegmented {len(segmented_meta)}/{len(debug_files)}")
print(f"Cutouts saved to: {CUTOUT_DIR}")

# Quick summary: how close did masks get to target area?
if segmented_meta:
    import pandas as pd
    _sm = pd.DataFrame(segmented_meta)
    if 'target_area' in _sm.columns and _sm['target_area'].notna().any():
        _sm['area_ratio'] = _sm['final_area'] / _sm['target_area']
        print("\nArea match summary (final_area / target_area):")
        print(_sm['area_ratio'].describe())
        within = ((_sm['area_ratio'] >= 0.5) & (_sm['area_ratio'] <= 2.0)).sum()
        print(f"Within 0.5x-2.0x of target: {within}/{len(_sm)} ({100*within/len(_sm):.0f}%)")


---
## Cell 10 — Batch Segmentation of All BIOSCAN Images

For production use: process the full CSV of BIOSCAN images.


In [ ]:
# ============================================================
# Batch segmentation: process ALL images, save cutouts to disk
# ============================================================
from tqdm.auto import tqdm
import gc

CUTOUT_DIR = OUTPUT_DIR / "cutouts"
CUTOUT_DIR.mkdir(exist_ok=True)

all_meta = []
image_files = sorted([p for p in BIOSCAN_DIR.glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f"Found {len(image_files)} images in {BIOSCAN_DIR}")
print(f"Cutouts will be saved to: {CUTOUT_DIR}")

_cls_map   = df.set_index('processid')['class'].to_dict()
_order_map2 = df.set_index('processid')['order'].to_dict()

for img_path in tqdm(image_files, desc="Segmenting"):
    processid = img_path.stem
    meta = metadata_lookup.get(processid, None)

    result = process_bioscan_image(
        img_path, predictor, mask_generator, metadata=meta, debug=False
    )
    if result is not None:
        cutout_path = CUTOUT_DIR / f"{processid}.png"
        rgba_bgra = cv2.cvtColor(result['rgba'], cv2.COLOR_RGBA2BGRA)
        cv2.imwrite(str(cutout_path), rgba_bgra)

        all_meta.append({
            'processid': processid,
            'cutout_path': str(cutout_path),
            'class': _cls_map.get(processid, 'Unknown'),
            'order': _order_map2.get(processid, 'Unknown'),
            'dish_radius': result['dish_radius'],
            'score': result['score'],
            'strategy': result['strategy'],
            'band':       result.get('band'),
            'tier':       result.get('tier'),
            'area_ratio': result.get('area_ratio'),
            'area_fraction': result['area_fraction'],
            'scale_factor': result['scale_factor'],
            'target_area': result.get('target_area'),
            'final_area':  result.get('final_area'),
        })
        del result

    if len(all_meta) % 100 == 0:
        gc.collect()

meta_df = pd.DataFrame(all_meta)
meta_csv = CUTOUT_DIR / "cutouts_metadata.csv"
meta_df.to_csv(meta_csv, index=False)

# Minimal quality.csv — filter cutouts at tray-generation time.
quality_cols = ['processid', 'cutout_path', 'band', 'tier', 'area_ratio',
                'target_area', 'final_area', 'strategy']
quality_df = meta_df[[c for c in quality_cols if c in meta_df.columns]]
quality_csv = CUTOUT_DIR / "quality.csv"
quality_df.to_csv(quality_csv, index=False)
print(f"Quality tags saved to: {quality_csv}")

gc.collect()
print(f"\nTotal segmented: {len(all_meta)} / {len(image_files)}")
print(f"Metadata saved to: {meta_csv}")

# Summary of area match quality
if 'target_area' in meta_df.columns and meta_df['target_area'].notna().any():
    valid = meta_df.dropna(subset=['target_area', 'final_area'])
    valid = valid[valid['target_area'] > 0]
    if len(valid) > 0:
        valid = valid.assign(area_ratio = valid['final_area'] / valid['target_area'])
        within = ((valid['area_ratio'] >= 0.5) & (valid['area_ratio'] <= 2.0)).sum()
        print(f"\nArea match: {within}/{len(valid)} within 0.5x-2.0x of target "
              f"({100*within/len(valid):.0f}%)")
        print(valid['area_ratio'].describe())


---
## Cell 11a — Learned Image Harmonization (PCTNet via libcom)

Replace classical relighting with a pretrained CNN that learns realistic compositing
from the iHarmony4 dataset (73k composite/real photo pairs).

Pipeline becomes: alpha-paste insect → crop around placement → PCTNet → paste back.
The model handles brightness, colour cast, contrast, and ambient bounce in one pass.


In [ ]:
# ============================================================
# IC-Light FBC — diffusion-based foreground relighting
# https://github.com/lllyasviel/IC-Light
# ============================================================
# Pipeline: SD1.5 base + IC-Light FBC weights (12-channel UNet conv_in)
# Inference: relight foreground to match a given background.
# We use it per-tray with a multiplicative-delta trick to keep full-res detail.
# ============================================================
import os, sys, subprocess
from pathlib import Path
import numpy as np
import cv2
import torch

ICLIGHT_DIR     = Path(os.environ.get("ICLIGHT_DIR", "./iclight_models")).resolve()
ICLIGHT_DIR.mkdir(parents=True, exist_ok=True)
ICLIGHT_DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
ICLIGHT_DTYPE   = torch.float16 if ICLIGHT_DEVICE == "cuda" else torch.float32

ICLIGHT_READY   = False
HARMONIZE_AVAILABLE = False

def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          check=False, capture_output=True, text=True)

# 1. Install dependencies
print("[iclight] installing diffusers / transformers / accelerate / safetensors …")
_pip("diffusers>=0.27.0", "transformers>=4.36.0", "accelerate", "safetensors", "huggingface_hub")

# 2. Download IC-Light FBC weights from HF
from huggingface_hub import hf_hub_download
fbc_w = ICLIGHT_DIR / "iclight_sd15_fbc.safetensors"
if not fbc_w.exists():
    print("[iclight] downloading IC-Light FBC weights (~1.5 GB) …")
    fbc_w = Path(hf_hub_download(
        repo_id="lllyasviel/ic-light",
        filename="iclight_sd15_fbc.safetensors",
        local_dir=str(ICLIGHT_DIR),
    ))

# 3. Load SD1.5 + patch UNet for IC-Light FBC (12 input channels)
import safetensors.torch as sf
from diffusers import (
    AutoencoderKL, UNet2DConditionModel,
    EulerAncestralDiscreteScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel

SD15 = "runwayml/stable-diffusion-v1-5"
print(f"[iclight] loading SD1.5 base from {SD15} (cached after first run) …")

try:
    tokenizer    = CLIPTokenizer.from_pretrained(SD15, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(SD15, subfolder="text_encoder",
                                                  torch_dtype=ICLIGHT_DTYPE).to(ICLIGHT_DEVICE)
    vae          = AutoencoderKL.from_pretrained(SD15, subfolder="vae",
                                                  torch_dtype=ICLIGHT_DTYPE).to(ICLIGHT_DEVICE)
    unet         = UNet2DConditionModel.from_pretrained(SD15, subfolder="unet",
                                                  torch_dtype=ICLIGHT_DTYPE).to(ICLIGHT_DEVICE)
    scheduler    = EulerAncestralDiscreteScheduler.from_pretrained(SD15, subfolder="scheduler")
except Exception as e:
    print(f"[iclight] SD1.5 download failed: {e}")
    print("[iclight] try: huggingface-cli login,  or set HF_HOME to a writable cache.")
    raise

# Patch UNet conv_in: 4 → 12 input channels (noise + comp_latent + bg_latent)
with torch.no_grad():
    old_conv = unet.conv_in
    new_conv = torch.nn.Conv2d(
        12, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
    ).to(ICLIGHT_DEVICE, ICLIGHT_DTYPE)
    new_conv.weight.zero_()
    new_conv.weight[:, :4, :, :].copy_(old_conv.weight)
    new_conv.bias.copy_(old_conv.bias)
    unet.conv_in = new_conv

# Load IC-Light FBC weights as additive offsets to UNet state
sd_off = sf.load_file(str(fbc_w))
unet_sd = unet.state_dict()
applied = 0
for k, v in sd_off.items():
    if k in unet_sd and unet_sd[k].shape == v.shape:
        unet_sd[k] = unet_sd[k] + v.to(unet_sd[k].device, unet_sd[k].dtype)
        applied += 1
unet.load_state_dict(unet_sd, strict=False)
print(f"[iclight] applied {applied}/{len(sd_off)} IC-Light FBC offset tensors to UNet.")

unet.eval(); vae.eval(); text_encoder.eval()
ICLIGHT_READY       = True
HARMONIZE_AVAILABLE = True
HARMONIZER_BACKEND  = "iclight-fbc"

# Cache encoded prompt (we always use the same one)
_DEFAULT_PROMPT = ("close-up overhead photo of dead insect specimens on a flat petri tray, "
                    "uniform softbox lighting, scientific imaging, sharp focus")
_NEG_PROMPT     = "blurry, low quality, painting, illustration, cartoon, watermark"

with torch.no_grad():
    def _embed(p):
        ids = tokenizer(p, padding="max_length", max_length=77,
                        truncation=True, return_tensors="pt").input_ids.to(ICLIGHT_DEVICE)
        return text_encoder(ids).last_hidden_state
    POS_EMB = _embed(_DEFAULT_PROMPT)
    NEG_EMB = _embed(_NEG_PROMPT)

print(f"[iclight] ready on {ICLIGHT_DEVICE} ({ICLIGHT_DTYPE}).")

VAE_SCALE = float(vae.config.scaling_factor)


# ── core IC-Light FBC inference ──
@torch.no_grad()
def _iclight_call(comp_u8, bg_u8, work_size=768,
                   num_steps=12, guidance_scale=2.0, seed=0):
    """Run IC-Light FBC at `work_size`. Returns uint8 RGB at work_size."""
    H0, W0 = comp_u8.shape[:2]
    comp = cv2.resize(comp_u8, (work_size, work_size), interpolation=cv2.INTER_AREA)
    bg   = cv2.resize(bg_u8,   (work_size, work_size), interpolation=cv2.INTER_AREA)

    def _to_t(arr):
        t = torch.from_numpy(arr).float() / 127.5 - 1.0
        return t.permute(2, 0, 1).unsqueeze(0).to(ICLIGHT_DEVICE, ICLIGHT_DTYPE)

    comp_t = _to_t(comp); bg_t = _to_t(bg)

    comp_lat = vae.encode(comp_t).latent_dist.mode() * VAE_SCALE
    bg_lat   = vae.encode(bg_t).latent_dist.mode()   * VAE_SCALE

    g = torch.Generator(device=ICLIGHT_DEVICE).manual_seed(seed)
    latents = torch.randn(comp_lat.shape, generator=g,
                           device=ICLIGHT_DEVICE, dtype=ICLIGHT_DTYPE)

    scheduler.set_timesteps(num_steps, device=ICLIGHT_DEVICE)
    latents = latents * scheduler.init_noise_sigma

    do_cfg = guidance_scale > 1.0
    pos_emb = POS_EMB; neg_emb = NEG_EMB
    if do_cfg:
        emb = torch.cat([neg_emb, pos_emb], dim=0)
        comp_lat_c = torch.cat([comp_lat, comp_lat], dim=0)
        bg_lat_c   = torch.cat([bg_lat,   bg_lat],   dim=0)

    for t in scheduler.timesteps:
        if do_cfg:
            x = torch.cat([latents, latents], dim=0)
            x = scheduler.scale_model_input(x, t)
            unet_in = torch.cat([x, comp_lat_c, bg_lat_c], dim=1)
            noise = unet(unet_in, t, encoder_hidden_states=emb).sample
            n_uc, n_c = noise.chunk(2)
            noise = n_uc + guidance_scale * (n_c - n_uc)
        else:
            x = scheduler.scale_model_input(latents, t)
            unet_in = torch.cat([x, comp_lat, bg_lat], dim=1)
            noise = unet(unet_in, t, encoder_hidden_states=pos_emb).sample
        latents = scheduler.step(noise, t, latents).prev_sample

    img = vae.decode(latents / VAE_SCALE).sample
    img = (img + 1.0).clamp(0, 2) * 127.5
    img = img.squeeze(0).permute(1, 2, 0).cpu().float().numpy().clip(0, 255).astype(np.uint8)
    return img  # work_size × work_size


# ── full-res application via multiplicative delta ──
def relight_tray_iclight(naive_full_rgb, bg_full_rgb, mask_full_u8,
                          work_size=768, smooth_sigma=2.0,
                          gain_clip=(0.45, 1.10), num_steps=12, seed=0):
    """Run IC-Light FBC on a downsampled tray, extract the per-pixel
    lighting gain it produced inside the foreground mask, smooth & upsample,
    apply that gain to the full-res naive composite. Detail preserved,
    lighting from the diffusion model."""
    H, W = naive_full_rgb.shape[:2]
    relit_lo = _iclight_call(naive_full_rgb, bg_full_rgb,
                              work_size=work_size, num_steps=num_steps, seed=seed)
    naive_lo = cv2.resize(naive_full_rgb, (work_size, work_size),
                           interpolation=cv2.INTER_AREA)

    # IC-Light desaturates if applied as RGB scale -> use L-only gain
    eps = 1.0
    relit_lab = cv2.cvtColor(relit_lo, cv2.COLOR_RGB2LAB).astype(np.float32)
    naive_lab = cv2.cvtColor(naive_lo, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_gain_lo = (relit_lab[..., 0] + eps) / (naive_lab[..., 0] + eps)
    L_gain_lo = np.clip(L_gain_lo, gain_clip[0], gain_clip[1])
    L_gain_lo = cv2.GaussianBlur(L_gain_lo, (0, 0), sigmaX=smooth_sigma)
    L_gain_full = cv2.resize(L_gain_lo, (W, H), interpolation=cv2.INTER_LINEAR)

    mask_f = mask_full_u8.astype(np.float32) / 255.0
    L_factor = (L_gain_full - 1.0) * mask_f + 1.0

    full_lab = cv2.cvtColor(naive_full_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    full_lab[..., 0] = np.clip(full_lab[..., 0] * L_factor, 0, 255)
    out = cv2.cvtColor(full_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    return out


# ── compatibility shim so generate_synthetic_tray keeps working ──
def composite_and_harmonize(*args, **kwargs):
    """Per-insect wrapper: kept for API compatibility but now a no-op alpha-paste.
    The real IC-Light pass happens once per tray after all insects are placed."""
    tray_rgb_np, insect_rgba_np, x, y = args[:4]
    return _alpha_paste_inplace(tray_rgb_np, insect_rgba_np, x, y)

def _alpha_paste_inplace(tray_rgb_np, insect_rgba_np, x, y):
    H, W = tray_rgb_np.shape[:2]
    ih, iw = insect_rgba_np.shape[:2]
    if x < 0 or y < 0 or x + iw > W or y + ih > H:
        cx1 = max(0, -x); cy1 = max(0, -y)
        cx2 = min(iw, W - x); cy2 = min(ih, H - y)
        if cx2 <= cx1 or cy2 <= cy1:
            return tray_rgb_np
        insect_rgba_np = insect_rgba_np[cy1:cy2, cx1:cx2]
        x = max(0, x); y = max(0, y)
        ih, iw = insect_rgba_np.shape[:2]
    a   = insect_rgba_np[..., 3:4].astype(np.float32) / 255.0
    rgb = insect_rgba_np[..., :3].astype(np.float32)
    target = tray_rgb_np[y:y+ih, x:x+iw].astype(np.float32)
    tray_rgb_np[y:y+ih, x:x+iw] = np.clip(
        a * rgb + (1.0 - a) * target, 0, 255).astype(np.uint8)
    return tray_rgb_np

print("Ready: relight_tray_iclight(naive, bg, mask) — call once per tray.")


---
## Cell 11 — Generate Synthetic Tray Images


In [ ]:
def load_cutout(meta_entry):
    """Load a single RGBA cutout from disk as PIL Image."""
    path = meta_entry['cutout_path']
    try:
        img = Image.open(path).convert("RGBA")
        return img
    except Exception:
        return None


def load_tray(tray_path):
    """Open a tray background, resized to TRAY_TARGET_WIDTH if one is set.

    Everything downstream takes its scale from this image's width, so this is the one
    place tray resolution is decided. See the note in the config cell.
    """
    im = Image.open(str(tray_path)).convert("RGBA")
    if TRAY_TARGET_WIDTH and im.width != TRAY_TARGET_WIDTH:
        _s = TRAY_TARGET_WIDTH / im.width
        im = im.resize((TRAY_TARGET_WIDTH, round(im.height * _s)),
                       Image.Resampling.LANCZOS)
    return im


def generate_synthetic_tray(
    tray_path,
    insect_meta_pool,
    num_insects,
):
    """
    Generate a single synthetic tray by compositing insects loaded from disk.
    Uses PIL alpha compositing (no Poisson blending) for clean, natural results.
    """
    tray_pil = load_tray(tray_path)
    tw, th = tray_pil.size
    placement_log = []
    occupied_regions = []

    margin_x = int(tw * PLACEMENT_MARGIN)
    margin_y = int(th * PLACEMENT_MARGIN)

    for i in range(num_insects):
        insect_data = random.choice(insect_meta_pool)

        # Load cutout from disk
        insect_pil = load_cutout(insect_data)
        if insect_pil is None:
            continue

        # Convert to numpy briefly for size calculation
        insect_np = np.array(insect_pil)

        # Calculate target size on tray
        target_h, target_w = calculate_insect_target_size(
            insect_np,
            insect_data['dish_radius'],
            (th, tw),
            insect_data['scale_factor'],
            insect_data['area_fraction']
        )
        del insect_np

        if target_h < 2 or target_w < 2:
            insect_pil.close()
            continue

        # Resize using PIL LANCZOS
        insect_pil = insect_pil.resize(
            (target_w, target_h),
            Image.Resampling.LANCZOS
        )

        # Random rotation
        angle = random.uniform(*ROTATION_RANGE)
        if abs(angle) > 0.5:
            insect_pil = insect_pil.rotate(
                angle,
                resample=Image.Resampling.BICUBIC,
                expand=True
            )

        iw, ih = insect_pil.size          # canvas: used for PLACEMENT and collisions

        # The RECORDED box must be the insect, not the canvas. expand=True grows the
        # canvas to hold the rotated rectangle and leaves its corners transparent, so
        # iw/ih overstate the insect by ~1.29x on average (4/pi for a uniform angle,
        # measured at 1.272x over 19,992 real boxes). Re-measure the alpha here.
        _ab = insect_pil.split()[-1].point(lambda v: 255 if v > 10 else 0).getbbox()
        if _ab is None:                   # fully transparent after rotation
            insect_pil.close()
            continue
        _bx0, _by0, _bx1, _by1 = _ab

        # Random placement with collision avoidance
        placed = False
        for attempt in range(50):
            x = random.randint(margin_x, max(margin_x, tw - iw - margin_x))
            y = random.randint(margin_y, max(margin_y, th - ih - margin_y))
            new_rect = (x, y, x + iw, y + ih)
            overlap = any(
                not (new_rect[2] < ox or new_rect[0] > ox2 or
                     new_rect[3] < oy or new_rect[1] > oy2)
                for ox, oy, ox2, oy2 in occupied_regions
            )
            if not overlap:
                placed = True
                break

        if not placed:
            x = random.randint(margin_x, max(margin_x, tw - iw - margin_x))
            y = random.randint(margin_y, max(margin_y, th - ih - margin_y))

        # Place insect on tray (PIL-based: edge defringe + shadow + alpha composite)
        tray_pil = place_insect_on_tray_pil(tray_pil, insect_pil, x, y)

        occupied_regions.append((x, y, x + iw, y + ih))
        placement_log.append({
            'processid': insect_data['processid'],
            'class': insect_data.get('class', 'Unknown'),
            'order': insect_data.get('order', 'Unknown'),
            # the insect's true extent, offset into tray coordinates
            'x': x + _bx0, 'y': y + _by0,
            'width': _bx1 - _bx0, 'height': _by1 - _by0,
            # the canvas box as pasted, kept for provenance
            'x_canvas': x, 'y_canvas': y, 'width_canvas': iw, 'height_canvas': ih,
            'angle': angle,
            'scale_factor': insect_data['scale_factor'],
            'area_fraction': insect_data['area_fraction'],
        })

        insect_pil.close()

    # Convert final tray back to BGR numpy for cv2.imwrite
    tray_rgb = np.array(tray_pil.convert("RGB"))
    tray_bgr = cv2.cvtColor(tray_rgb, cv2.COLOR_RGB2BGR)
    tray_pil.close()

    return tray_bgr, placement_log


print("Tray generation function defined (PIL-based compositing, disk cutouts).")


---
## Cell 12 — Run Generation!


In [ ]:
# ============================================================
# PREFLIGHT — run this immediately before the generation cell
# ============================================================
# Generation is a multi-hour job. Everything it needs is checked here first, so a
# missing name or a wrong path costs a second rather than an afternoon.
#
# CELLS TO RUN, in this order:
#      5   config and paths
#      7   specimen table (df, metadata_lookup)
#     13   calculate_insect_target_size
#     15   place_insect_on_tray_pil + blending helpers
#     25   generate_synthetic_tray (clean)
#     27   PREFLIGHT
#  -> 28   GENERATE clean trays   [hours]
#     36   apply_soil / apply_soil_staining
#     37   generate_synthetic_tray_brown
#     39   brown visual test (optional)
#  -> 41   GENERATE brown trays   [hours]
#  -> 43   export to COCO
#     45   VERIFY the export
#
# Skip every other cell. 9, 11, 17, 19 and 21 are the SAM-2 segmentation pipeline
# and 23 is IC-Light — the cutouts already exist, and nothing in the list above
# needs a GPU or those downloads.
import sys
from PIL import Image
_fail, _warn = [], []

for _n in ("df", "metadata_lookup", "calculate_insect_target_size",
           "place_insect_on_tray_pil", "extend_edge_colors", "add_shadow_pil",
           "blend_insect_edges_with_background", "load_cutout",
           "generate_synthetic_tray"):
    if _n not in dir() and _n not in globals():
        _fail.append(f"{_n} is not defined — run the cell that defines it "
                     f"(see the list above)")

for _p, _what in ((TRAY_DIR, "tray backgrounds"), (CUTOUT_DIR, "cutouts"),
                  (OUTPUT_DIR, "output folder")):
    if not Path(_p).is_dir(): _fail.append(f"{_what}: {_p} is not a directory")

# OUTPUT_DIR is on a network mount — prove it is writable now, not in three hours
try:
    _probe = OUTPUT_DIR / ".write_probe"
    _probe.write_bytes(b"x" * 1024)
    _probe.unlink()
    print(f"write:   {OUTPUT_DIR} is writable")
except Exception as _e:
    _fail.append(f"cannot write to {OUTPUT_DIR}: {_e}")

_meta = CUTOUT_DIR / "cutouts_metadata.csv"
if not _meta.exists():
    _npng = len(list(CUTOUT_DIR.glob("*.png"))) if CUTOUT_DIR.is_dir() else 0
    _fail.append(
        f"no cutouts_metadata.csv at {_meta}\n"
        f"      (that folder holds {_npng:,} .png files)\n"
        f"      Cell 5 prints every location it searched. Find the CSV with:\n"
        f"          find {BASE_DIR.resolve()} -name cutouts_metadata.csv\n"
        f"      then set CUTOUT_DIR in cell 5 to the folder containing it. The PNGs and\n"
        f"      the CSV must be in the SAME folder — the CSV's own cutout_path column is\n"
        f"      re-anchored onto CUTOUT_DIR, so stale absolute paths in it are fine.")
_bl = BASE_DIR / "PRUNE_REVIEW" / "cutout_blocklist.csv"
if not _bl.exists():
    _fail.append(f"no prune blocklist at {_bl.resolve()} — without it, 21 specimens "
                 f"whose order is not in CLASS_NAMES go back into the trays")

_ntray = len(sorted(TRAY_DIR.rglob("*.png")))
if _ntray == 0: _fail.append(f"no .png tray backgrounds under {TRAY_DIR}")

# Is anything already in the output folder? Re-running appends rather than replaces,
# and the export globs whatever it finds.
_old = list(OUTPUT_DIR.glob("synthetic_tray_*")) + list(OUTPUT_DIR.glob("brown_tray_*"))
if _old:
    _warn.append(f"{len(_old)} tray images are ALREADY in {OUTPUT_DIR}. Generation "
                 f"overwrites by index but does not clear the folder, so a shorter run "
                 f"leaves older trays behind and the export will pick them up. Delete "
                 f"them first if this is a fresh run.")

# Disk. This is the constraint that actually bites: at 2644x2108 a lossless PNG tray
# measures ~14.5 MB, and the COCO export then writes a JPEG copy of every one at
# ~2.8 MB. Both sets exist at the same time, so the peak is the sum. Measured, not
# guessed. PNG compression level makes no difference — photographic noise does not
# compress — so the only lever is the intermediate format.
try:
    import shutil as _sh
    _ntot  = int(round(NUM_TRAYS_TO_GENERATE * 1.35))   # clean + the 35% brown subset
    # per-tray size scales with pixel count, so account for any tray resize
    _px_mult = 1.0
    if _ntray:
        _w0 = Image.open(sorted(TRAY_DIR.rglob("*.png"))[0]).width
        if TRAY_TARGET_WIDTH:
            _px_mult = (TRAY_TARGET_WIDTH / _w0) ** 2
    _per   = (0.0145 if TRAY_FORMAT == "png" else 0.0039) * _px_mult
    _png   = _per * _ntot
    _jpg   = 0.0028 * _px_mult * _ntot
    _peak  = _png + _jpg
    _free  = _sh.disk_usage(OUTPUT_DIR).free / 1e9
    print(f"disk:    {_free:,.0f} GB free | {_ntot:,} trays as {TRAY_FORMAT.upper()} "
          f"~{_png:,.0f} GB + ~{_jpg:,.0f} GB exported JPEG = ~{_peak:,.0f} GB peak")
    if _free < _peak:
        _msg = (f"only {_free:,.0f} GB free at {OUTPUT_DIR}, and the run peaks at about "
                f"{_peak:,.0f} GB ({_png:,.0f} GB of {TRAY_FORMAT.upper()} trays + "
                f"{_jpg:,.0f} GB of exported JPEGs, both on disk at once).")
        if TRAY_FORMAT == "png":
            _msg += (f"\n      Setting TRAY_FORMAT = 'jpg' in cell 5 brings the peak to "
                     f"~{0.0067*_ntot:,.0f} GB, at a measured cost of 0.07 dB in the "
                     f"exported image.")
        else:
            _msg += (f"\n      Already on JPEG trays, so the remaining levers are freeing "
                     f"~{_peak-_free:,.0f} GB or lowering NUM_TRAYS_TO_GENERATE to about "
                     f"{int(_free/ (_peak/NUM_TRAYS_TO_GENERATE) * 0.9):,}.")
        _fail.append(_msg)
    elif _free < _peak * 1.15:
        _warn.append(f"{_free:,.0f} GB free against a ~{_peak:,.0f} GB peak — under 15% "
                     f"headroom. A second run, or anything else writing to this disk, "
                     f"will fill it.")
except Exception as _e:
    print(f"disk: could not be checked ({_e})")

print(f"trays:   {_ntray} backgrounds under {TRAY_DIR.resolve()}")
if _ntray:
    from collections import Counter as _C
    _sz = _C(Image.open(_f).size for _f in list(TRAY_DIR.rglob("*.png"))[:40])
    (_w, _h), _n = _sz.most_common(1)[0]
    _weff = TRAY_TARGET_WIDTH or _w
    _heff = round(_h * _weff / _w)
    print(f"         source {_w}x{_h} ({_w/TRAY_WIDTH_MM:.2f} px/mm)"
          + (f"  ->  generated at {_weff}x{_heff} "
             f"({_weff/TRAY_WIDTH_MM:.2f} px/mm)" if TRAY_TARGET_WIDTH else "")
          + f"   [real 3DTRAB 10.85 px/mm]")

    # Does the upscale actually buy detail? It only does if the CUTOUTS are large
    # enough that each insect is still being DOWNscaled onto the tray. If a cutout is
    # smaller than its target size, the insect is interpolated and the extra tray
    # pixels are empty magnification. Sample the pool and check.
    if TRAY_TARGET_WIDTH and _meta.exists():
        import random as _r
        _mm = pd.read_csv(_meta).to_dict("records")
        _mm = [m for m in _mm if (CUTOUT_DIR / f"{m['processid']}.png").exists()]
        _up, _tot_c, _ratios = 0, 0, []
        for _m in _r.Random(0).sample(_mm, min(150, len(_mm))):
            try:
                with Image.open(CUTOUT_DIR / f"{_m['processid']}.png") as _ci:
                    _cw, _ch = _ci.size
                _th_, _tw_ = calculate_insect_target_size(
                    np.zeros((_ch, _cw, 4), np.uint8), _m["dish_radius"],
                    (_heff, _weff), _m["scale_factor"], _m["area_fraction"])
            except Exception:
                continue
            _tot_c += 1
            _ratios.append(max(_tw_, _th_) / max(_cw, _ch))
            if max(_tw_, _th_) > max(_cw, _ch): _up += 1
        if _tot_c:
            _med = sorted(_ratios)[len(_ratios)//2]
            print(f"         cutouts: median render is {_med:.2f}x the cutout's own size"
                  f" ({_up}/{_tot_c} would be upscaled)")
            if _up > 0.25 * _tot_c:
                _warn.append(
                    f"{_up/_tot_c:.0%} of sampled insects would be rendered LARGER than "
                    f"their cutout at {_weff} px wide, i.e. interpolated. The upscale "
                    f"then adds tray pixels without adding insect detail. Consider "
                    f"TRAY_TARGET_WIDTH = None.")
print(f"cutouts: {CUTOUT_DIR}")
print(f"output:  {OUTPUT_DIR}")
print(f"export:  {NUM_TRAYS_TO_GENERATE:,} clean + "
      f"{int(round(NUM_TRAYS_TO_GENERATE*0.35)):,} brown trays at FULL resolution "
      f"(EXPORT_MAX_SIDE is set in the export cell)")

for _w in _warn: print(f"\nWARNING: {_w}")
if _fail:
    print("\nNOT READY:")
    for _f in _fail: print(f"  - {_f}")
    raise RuntimeError(f"{len(_fail)} problem(s) above — fix before generating")
print("\nPREFLIGHT OK — safe to run the generation cell.")


In [ ]:
# ── Load cutout metadata from disk if not already in memory (OR if empty) ──
try:
    _have_meta = len(all_meta) > 0        # exists AND non-empty
except NameError:
    _have_meta = False

if _have_meta:
    print(f"Using in-memory all_meta ({len(all_meta)} entries)")
else:
    meta_csv = CUTOUT_DIR / "cutouts_metadata.csv"
    if meta_csv.exists():
        _df = pd.read_csv(meta_csv)
        all_meta = _df.to_dict('records')
        del _df
        # cutout_path was written as an ABSOLUTE path by the segmentation cell, so a
        # metadata CSV copied from another folder (or another machine) points at
        # files that are not there. Re-anchor on CUTOUT_DIR before testing existence,
        # otherwise every row silently vanishes and the pool comes out empty.
        _relinked = 0
        for _m in all_meta:
            if not Path(_m['cutout_path']).exists():
                _alt = CUTOUT_DIR / Path(str(_m['cutout_path'])).name
                if _alt.exists():
                    _m['cutout_path'] = str(_alt); _relinked += 1
        _before = len(all_meta)
        all_meta = [m for m in all_meta if Path(m['cutout_path']).exists()]
        print(f"Loaded all_meta from {meta_csv} "
              f"({len(all_meta)}/{_before} cutouts found on disk"
              + (f", {_relinked} re-anchored on {CUTOUT_DIR}" if _relinked else "") + ")")
        assert len(all_meta) > 0.5 * _before, (
            f"only {len(all_meta)} of {_before} cutout files were found. The metadata "
            f"CSV points somewhere else and re-anchoring on {CUTOUT_DIR} did not help "
            "— check that CUTOUT_DIR is the folder that actually holds the .png cutouts.")
    else:
        raise FileNotFoundError(
            f"No cutouts_metadata.csv found at {meta_csv}. "
            "Run the batch segmentation cell first (or check OUTPUT_DIR/CUTOUT_DIR path)."
        )


# ── prune blocklist ────────────────────────────────────────────────────────────
ALLOW_MISSING_BLOCKLIST = False   # set True only to deliberately generate unpruned
# Applied HERE, outside the load branch above, because that branch is skipped when
# all_meta is already in memory from an earlier cell — which would silently
# generate with every rejected specimen still in the pool.
_bl = BASE_DIR / "PRUNE_REVIEW" / "cutout_blocklist.csv"
if _bl.exists():
    _block = set(pd.read_csv(_bl)["processid"].astype(str))
    _before = len(all_meta)
    all_meta = [m for m in all_meta if str(m["processid"]) not in _block]
    print(f"blocklist {_bl.name}: pool {_before} -> {len(all_meta)} "
          f"({_before - len(all_meta)} of {len(_block)} listed specimens removed)")
    _still = {str(m["processid"]) for m in all_meta} & _block
    assert not _still, f"blocklisted specimens still in the pool: {sorted(_still)[:5]}"
elif ALLOW_MISSING_BLOCKLIST:
    print(f"NO BLOCKLIST at {_bl} — generating with every cutout (explicitly allowed)")
else:
    raise FileNotFoundError(
        f"No blocklist at {_bl}.\n\n"
        "Generating without it silently puts back the 21 specimens whose order is "
        "NOT in CLASS_NAMES (Julida, Chordeumatida, Polydesmida, Geophilomorpha, "
        "Raphidioptera, Siphonaptera) plus the 4 fragment photographs. The rare-order "
        "recipes give those specimens far MORE tray time than their count suggests, "
        "so this is not a rounding error.\n\n"
        "Fix: run inspect_and_prune.ipynb to write it, or check BASE_DIR — this path "
        f"is resolved relative to BASE_DIR = {BASE_DIR.resolve()}.\n"
        "To generate without pruning on purpose, set ALLOW_MISSING_BLOCKLIST = True.")

# the pool must be non-empty and one row per specimen, or the disjoint split is a lie
assert all_meta, (
    "the specimen pool is EMPTY after loading and blocklisting. The usual cause is "
    f"cutout_path in cutouts_metadata.csv pointing somewhere that no longer exists — "
    f"check that the paths in {CUTOUT_DIR/'cutouts_metadata.csv'} resolve on this machine.")
_pids = [str(m["processid"]) for m in all_meta]
_dupe = {p for p in _pids if _pids.count(p) > 1} if len(_pids) != len(set(_pids)) else set()
assert not _dupe, (
    f"{len(_dupe)} processids appear on more than one cutout, e.g. {sorted(_dupe)[:5]}. "
    "The split below partitions CUTOUTS while claiming to partition SPECIMENS — with "
    "duplicates the same individual can land in both train and test.")
print(f"pool: {len(all_meta)} cutouts, one per specimen")

# ── Quality filter: drop only SEVERE area-misses ──
SEVERE_LO, SEVERE_HI = 0.30, 4.0

def _keep(m):
    r = m.get('area_ratio')
    try:
        r = float(r)
        if r != r:
            return True
    except (TypeError, ValueError):
        return True
    return SEVERE_LO <= r <= SEVERE_HI

_before = len(all_meta)
all_meta_filtered = [m for m in all_meta if _keep(m)]
_dropped = _before - len(all_meta_filtered)
print(f"Quality filter: kept {len(all_meta_filtered)}/{_before} cutouts "
      f"(dropped {_dropped} with area_ratio outside [{SEVERE_LO}, {SEVERE_HI}])")

# ============================================================
# STRATIFIED DISTRIBUTION SYSTEM  (specimen-level DISJOINT split)
# ============================================================
from itertools import combinations as _combinations
from tqdm.auto import tqdm

# ── Step 1: join order info from CSV (in-memory only) ──
# df was loaded earlier from selected_classes_original_5000.csv
_order_map = df.set_index('processid')['order'].to_dict()
for _m in all_meta_filtered:
    if 'order' not in _m:
        _m['order'] = _order_map.get(_m['processid'], 'Unknown')

# ── Step 2: build per-order pools (FULL, before the split) ──
class_pools_full = {}
for _m in all_meta_filtered:
    _ord = _m.get('order', 'Unknown')
    if _ord and _ord != 'Unknown':
        class_pools_full.setdefault(_ord, []).append(_m)
class_pools_full = dict(sorted(class_pools_full.items(), key=lambda kv: -len(kv[1])))

# ── Step 2b: SPECIMEN-LEVEL DISJOINT SPLIT ───────────────────────────────
# Partition EACH order's specimens (cutouts) into disjoint train/valid/test
# pools BEFORE any tray is generated. A specimen assigned to `train` will
# NEVER appear in a valid/test tray, so evaluation measures genuine
# generalisation to UNSEEN specimens (eliminates the train/test leakage that
# previously inflated rare-order AP).
SPECIMEN_SPLIT = (0.60, 0.20, 0.20)   # train / valid / test, by SPECIMEN
random.seed(RANDOM_SEED if RANDOM_SEED is not None else 42)

split_pools = {'train': {}, 'valid': {}, 'test': {}}
_specimen_counts = {}   # order -> (n_train, n_valid, n_test)

for _ord, _items in class_pools_full.items():
    _it = list(_items)
    random.shuffle(_it)
    _n = len(_it)
    if _n == 0:
        continue
    if _n == 1:
        # single specimen: can only go to train (NOT evaluable)
        _parts = {'train': _it[:], 'valid': [], 'test': []}
    elif _n == 2:
        # 1 train, 1 test — disjoint, no val (honest but high-variance)
        _parts = {'train': _it[:1], 'valid': [], 'test': _it[1:]}
    else:
        _ntr = max(1, int(round(_n * SPECIMEN_SPLIT[0])))
        _nva = max(1, int(round(_n * SPECIMEN_SPLIT[1])))
        if _ntr + _nva >= _n:               # guarantee >= 1 test specimen
            _nva = max(1, _n - _ntr - 1)
            if _ntr + _nva >= _n:
                _ntr, _nva = _n - 2, 1
        _parts = {
            'train': _it[:_ntr],
            'valid': _it[_ntr:_ntr + _nva],
            'test':  _it[_ntr + _nva:],
        }
    for _sp in ('train', 'valid', 'test'):
        if _parts[_sp]:
            split_pools[_sp][_ord] = _parts[_sp]
    _specimen_counts[_ord] = (len(_parts['train']), len(_parts['valid']), len(_parts['test']))

print(f"\nSpecimen-level DISJOINT split ({len(class_pools_full)} orders):")
print(f"  {'order':25s} {'train':>6s} {'valid':>6s} {'test':>6s}")
for _ord in class_pools_full:
    _tr, _va, _te = _specimen_counts.get(_ord, (0, 0, 0))
    _flag = ('  <- 1 test specimen (noisy AP)' if _te == 1
             else ('  <- NOT evaluable (train-only)' if _te == 0 else ''))
    print(f"  {_ord:25s} {_tr:6d} {_va:6d} {_te:6d}{_flag}")

# ── Step 3: weighted-pool builder (reads the CURRENT-split `class_pools`) ──
# `class_pools` is rebound to split_pools[<split>] inside the generation loop,
# so a pool built here only ever contains specimens from the active split.
_POOL_SIZE = 300  # >> INSECTS_PER_TRAY max; ensures stable proportions

def _build_pool(fracs):
    """Return a shuffled list where class proportions match `fracs`.
    Samples from the global `class_pools` (rebound per split below)."""
    pool = []
    for _o, _frac in fracs.items():
        _src = class_pools.get(_o, [])
        if not _src:
            continue
        _n = max(1, round(_frac * _POOL_SIZE))
        # sample without replacement up to pool size; top up w/ replacement for small pools
        pool += random.sample(_src, min(_n, len(_src)))
        if _n > len(_src):
            pool += random.choices(_src, k=_n - len(_src))
    if not pool:   # safety: never hand generate_synthetic_tray an empty pool
        _flat = [m for ms in class_pools.values() for m in ms]
        if _flat:
            pool = random.choices(_flat, k=_POOL_SIZE)
    random.shuffle(pool)
    return pool

# ── Step 4: recipe functions ──
# Each returns {order: fraction} with values that sum to 1.0.
# They read the CURRENT-split tier lists (RICH_ORDERS, ...) at call time.

def _pick(pool, n=1):
    """Pick n unique items from pool (Python random, no replacement)."""
    return random.sample(pool, min(n, len(pool)))

def _recipe_mono():
    """100 % single rich class — model sees one species in isolation."""
    return {_pick(RICH_ORDERS or all_orders)[0]: 1.0}

def _recipe_dominant_2():
    """70/30 split — one species dominates, one is minority."""
    a, b = _pick(all_orders, 2)
    return {a: 0.70, b: 0.30}

def _recipe_dominant_3():
    """60/25/15 — one dominant, two supporting."""
    picks = _pick(all_orders, min(3, len(all_orders)))
    bases = [0.60, 0.25, 0.15][:len(picks)]
    _s = sum(bases)
    return {o: b/_s for o, b in zip(picks, bases)}

def _recipe_equal_2():
    """50/50 pair — model must distinguish two co-occurring species."""
    a, b = _pick(all_orders, 2)
    return {a: 0.50, b: 0.50}

def _recipe_equal_3():
    """Three species at equal density."""
    picks = _pick(all_orders, min(3, len(all_orders)))
    return {o: 1/len(picks) for o in picks}

def _recipe_equal_5():
    """Five species at equal density."""
    n = min(5, len(all_orders))
    picks = _pick(all_orders, n)
    return {o: 1/n for o in picks}

def _recipe_equal_8():
    """Eight species (rich+medium pool) at equal density."""
    _pool = RICH_ORDERS + MEDIUM_ORDERS
    n = min(8, len(_pool))
    picks = _pick(_pool, n)
    return {o: 1/n for o in picks}

def _recipe_rich_all():
    """All rich classes at equal density — maximum inter-class context."""
    picks = RICH_ORDERS or all_orders[:11]
    return {o: 1/len(picks) for o in picks}

def _recipe_rare_spotlight():
    """One sparse/rare class at 30 %, three rich classes share the rest.
    Ensures rare orders appear in enough images of this split."""
    _candidates = SPARSE_ORDERS + RARE_ORDERS
    if not _candidates:
        return _recipe_equal_3()
    rare = _pick(_candidates)[0]
    companions = _pick([x for x in RICH_ORDERS + MEDIUM_ORDERS if x != rare],
                       min(3, len(RICH_ORDERS + MEDIUM_ORDERS)))
    fracs = {rare: 0.30}
    for _o in companions:
        fracs[_o] = 0.70 / max(1, len(companions))
    return fracs

def _recipe_frequency_weighted():
    """5-8 classes sampled proportionally to pool size (within this split)."""
    _n = random.randint(5, min(8, len(all_orders)))
    _w = np.array([len(class_pools[o]) for o in all_orders], dtype=float)
    _w /= _w.sum()
    picks = np.random.choice(all_orders, size=_n, replace=False, p=_w).tolist()
    _fw = np.array([len(class_pools[o]) for o in picks], dtype=float)
    _fw /= _fw.sum()
    return dict(zip(picks, _fw.tolist()))

_RECIPES = [
    (_recipe_mono,               0.08),
    (_recipe_dominant_2,         0.12),
    (_recipe_dominant_3,         0.10),
    (_recipe_equal_2,            0.12),
    (_recipe_equal_3,            0.12),
    (_recipe_equal_5,            0.12),
    (_recipe_equal_8,            0.10),
    (_recipe_rich_all,           0.08),
    (_recipe_rare_spotlight,     0.10),
    (_recipe_frequency_weighted, 0.06),
]

# ── Step 5: schedule builder (per split) ──
def _build_schedule(n_target):
    """Build a generation schedule of length ~n_target using the CURRENT-split
    tier lists. Systematic phase guarantees coverage of every class & rich pair
    present in the split; random phase fills the rest with recipe variety."""
    _schedule = []
    # (a) 2 mono trays per rich class
    for _o in RICH_ORDERS:
        _schedule += [{_o: 1.0}, {_o: 1.0}]
    # (b) 1 paired tray per unique rich-class pair
    for _o1, _o2 in _combinations(RICH_ORDERS, 2):
        _schedule.append({_o1: 0.50, _o2: 0.50})
    # (c) 2 spotlight trays per non-rich class — guarantees rare classes appear
    for _o in MEDIUM_ORDERS + SPARSE_ORDERS + RARE_ORDERS:
        _comp = _pick([x for x in RICH_ORDERS + MEDIUM_ORDERS if x != _o],
                      min(3, len(RICH_ORDERS)))
        _fracs = {_o: 0.35}
        for _c in _comp:
            _fracs[_c] = 0.65 / max(1, len(_comp))
        _schedule += [dict(_fracs), dict(_fracs)]
    _systematic_min = len(_schedule)
    # (d) random variety up to n_target
    _rfns, _rw = zip(*_RECIPES)
    while len(_schedule) < n_target:
        _schedule.append(random.choices(_rfns, weights=_rw, k=1)[0]())
    random.shuffle(_schedule)
    if n_target >= _systematic_min:
        _schedule = _schedule[:n_target]
    return _schedule, _systematic_min

# ── Step 6: generate trays PER SPLIT from the disjoint specimen pools ──
# Trays are written with the split encoded in the filename so the COCO export
# can assign them without any further (leak-prone) random slicing:
#   synthetic_tray_<split>_####.png   (+ matching _placements.csv)
TRAY_SPLIT_FRACS = {'train': 0.60, 'valid': 0.20, 'test': 0.20}
tray_paths = sorted(TRAY_DIR.rglob('*.png'))
print(f"\nAvailable tray backgrounds: {len(tray_paths)}")
print(f"Insects per tray: {INSECTS_PER_TRAY[0]}-{INSECTS_PER_TRAY[1]}")
print("=" * 60)

all_placement_logs = []
_class_tally = {}

for _split in ('train', 'valid', 'test'):
    # Rebind the globals the recipes / pool-builder read, to THIS split's pools.
    class_pools  = split_pools[_split]
    all_orders   = list(class_pools.keys())
    RICH_ORDERS   = [o for o in all_orders if len(class_pools[o]) >= 200]
    MEDIUM_ORDERS = [o for o in all_orders if 50 <= len(class_pools[o]) < 200]
    SPARSE_ORDERS = [o for o in all_orders if 10 <= len(class_pools[o]) < 50]
    RARE_ORDERS   = [o for o in all_orders if 0 < len(class_pools[o]) < 10]

    _n_trays = int(round(NUM_TRAYS_TO_GENERATE * TRAY_SPLIT_FRACS[_split]))
    _schedule, _sysmin = _build_schedule(_n_trays)
    print(f"\n[{_split}] {len(_schedule)} trays (systematic min {_sysmin}) | "
          f"{len(all_orders)} orders | "
          f"{sum(len(v) for v in class_pools.values())} specimens")

    for _i, _fracs in enumerate(tqdm(_schedule, desc=f"{_split} trays")):
        tray_path   = random.choice(tray_paths)
        num_insects = random.randint(*INSECTS_PER_TRAY)
        pool        = _build_pool(_fracs)

        result_tray, placement_log = generate_synthetic_tray(
            tray_path=tray_path,
            insect_meta_pool=pool,
            num_insects=num_insects,
        )

        output_name = f"synthetic_tray_{_split}_{_i + 1:04d}{TRAY_EXT}"
        output_path = OUTPUT_DIR / output_name
        cv2.imwrite(str(output_path), result_tray, _TRAY_PARAMS)
        del result_tray

        pd.DataFrame(placement_log).to_csv(
            OUTPUT_DIR / f"synthetic_tray_{_split}_{_i + 1:04d}_placements.csv",
            index=False)

        for _entry in placement_log:
            _o = _order_map.get(_entry.get('processid', ''), 'Unknown')
            _class_tally[_o] = _class_tally.get(_o, 0) + 1

        all_placement_logs.append({
            'split':       _split,
            'tray_index':  _i + 1,
            'background':  tray_path.name,
            'num_insects': len(placement_log),
            'output_path': str(output_path),
        })

        if (_i + 1) % 25 == 0:
            gc.collect()

    print(f"  [{_split}] done: {len(_schedule)} trays -> {OUTPUT_DIR}")

print("\n" + "=" * 60)
print(f"Generation complete! {len(all_placement_logs)} clean trays -> {OUTPUT_DIR}")
print("\nDataset-level class distribution (all splits, clean trays):")
for _o, _cnt in sorted(_class_tally.items(), key=lambda x: -x[1]):
    if _cnt > 0:
        print(f"  {_o:25s}: {_cnt:5d} insects across all trays")


---
## Cell 13 — Visualize Results


In [ ]:
# Display a grid of generated trays
generated_trays = sorted(OUTPUT_DIR.glob('synthetic_tray_*.png'))
num_show = min(4, len(generated_trays))

if num_show > 0:
    fig, axes = plt.subplots(1, num_show, figsize=(6 * num_show, 5))
    if num_show == 1:
        axes = [axes]

    for ax, tray_path in zip(axes, generated_trays[:num_show]):
        img = cv2.imread(str(tray_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(tray_path.name, fontsize=10)
        ax.axis('off')

    plt.suptitle("Generated Synthetic Tray Images", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No generated trays found. Run Cell 12 first.")


---
## Cell 14 — Side-by-Side Comparison: Original vs Synthetic


In [ ]:
# Compare an original tray with a synthetic version
if len(generated_trays) > 0 and len(tray_paths) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Original tray
    orig = cv2.imread(str(tray_paths[0]))
    axes[0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original: {tray_paths[0].name}", fontsize=12)
    axes[0].axis('off')

    # Synthetic tray
    synth = cv2.imread(str(generated_trays[0]))
    axes[1].imshow(cv2.cvtColor(synth, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Synthetic: {generated_trays[0].name}", fontsize=12)
    axes[1].axis('off')

    plt.suptitle("Original vs Synthetic Tray", fontsize=14)
    plt.tight_layout()
    plt.show()

# Print summary statistics
print("\nGeneration Summary:")
summary_df = pd.DataFrame(all_placement_logs)
print(summary_df.to_string(index=False))


---
## Cell 15 — Zoom Into Detail: Verify Insect Sizing & Blending Quality


In [ ]:
# Zoom into a placed insect to inspect blending quality
if len(generated_trays) > 0 and len(all_placement_logs) > 0:
    # Load the first generated tray and its placement log
    tray_img = cv2.imread(str(generated_trays[0]))
    log_path = OUTPUT_DIR / f"synthetic_tray_0001_placements.csv"

    if log_path.exists():
        log_df = pd.read_csv(log_path)

        # Show zoomed views of up to 6 placed insects
        num_zoom = min(6, len(log_df))
        fig, axes = plt.subplots(2, num_zoom, figsize=(4 * num_zoom, 8))

        for i in range(num_zoom):
            row = log_df.iloc[i]
            x, y, w, h = int(row['x']), int(row['y']), int(row['width']), int(row['height'])

            # Add context padding
            pad = max(w, h)
            x1 = max(0, x - pad)
            y1 = max(0, y - pad)
            x2 = min(tray_img.shape[1], x + w + pad)
            y2 = min(tray_img.shape[0], y + h + pad)

            crop = tray_img[y1:y2, x1:x2]
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            # Wide view
            axes[0][i].imshow(crop_rgb)
            axes[0][i].set_title(f"{row['processid']}\nsf={row['scale_factor']:.3f}", fontsize=9)
            axes[0][i].axis('off')

            # Tight crop
            tight = tray_img[y:min(y+h, tray_img.shape[0]), x:min(x+w, tray_img.shape[1])]
            axes[1][i].imshow(cv2.cvtColor(tight, cv2.COLOR_BGR2RGB))
            axes[1][i].set_title(f"{w}x{h} px\n~{row['scale_factor']*8:.1f}mm", fontsize=9)
            axes[1][i].axis('off')

        plt.suptitle("Zoom: Placed Insects — Context (top) & Tight Crop (bottom)", fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("Placement log not found.")


---
## Cell 17 — Brown / Soil-Stained Augmentation Subset

Mimics specimens recovered from a soil / pitfall trap (see reference photo): insects coated in mud (brown cast, desaturated, darkened), small soil specks stuck to bodies, a faint wet sheen, plus scattered soil debris and translucent tan water-stains on the tray.

Two things the clean dataset lacks are added here:
- **Clustering / pairing** — insects are placed in tight groups that touch and overlap, so the model learns clumped, partially-occluded specimens.
- **Soil debris as hard negatives** — unlabelled specks teach the detector to ignore soil clumps that look like tiny insects (fewer false positives).

Flow: **functions → generator → TEST cell (visual check) → batch**. Brown trays are written to `OUTPUT_DIR` as `brown_tray_####.png` and are picked up automatically by the COCO export.


In [ ]:
# ============================================================
# SOIL / BROWN AUGMENTATION TOOLKIT
# ------------------------------------------------------------
# Mimics specimens recovered from a pitfall / soil trap: insects coated
# in mud (brown colour cast + desaturated + darkened), small soil specks
# stuck to their bodies, a faint wet sheen, plus scattered soil debris
# and translucent tan water-stains on the tray itself.
# ============================================================
import numpy as np
import cv2
from PIL import Image

# Representative soil / mud tones (RGB). Sampled at random per insect.
SOIL_TONES = [
    (96, 67, 40),    # dark brown
    (120, 85, 52),   # medium brown
    (140, 105, 70),  # tan brown
    (78, 56, 38),    # deep umber
    (110, 78, 45),   # earthy
]
MUD_STAIN_TONE = (168, 128, 82)   # translucent tan (soil-stained water)


def apply_soil_staining(insect_rgba_pil, intensity=None, rng=None):
    """Return an RGBA cutout tinted toward muddy brown, desaturated, darkened,
    speckled with soil and given a faint wet sheen. `intensity` in [0,1]
    controls how heavily the insect is coated (default: random 0.35-0.65)."""
    rng = rng if rng is not None else np.random.default_rng()
    if intensity is None:
        intensity = float(rng.uniform(0.35, 0.65))

    arr   = np.array(insect_rgba_pil).astype(np.float32)
    rgb   = arr[..., :3]
    alpha = arr[..., 3]
    body  = alpha > 10
    if body.sum() < 5:
        return insect_rgba_pil

    h, w = alpha.shape

    # 1. Brown colour cast — blend toward a random soil tone
    tone = np.array(SOIL_TONES[int(rng.integers(len(SOIL_TONES)))], dtype=np.float32)
    stained = (1.0 - intensity) * rgb + intensity * tone

    # 2. Desaturate (soil mutes the insect's own colour)
    hsv = cv2.cvtColor(np.clip(stained, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] *= (1.0 - 0.40 * intensity)
    stained = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)

    # 3. Slight darkening
    stained *= (1.0 - 0.15 * intensity)

    # 4. Soil specks — small dark dots stuck to the body
    ys, xs = np.where(body)
    n_specks = int(np.clip(np.sqrt(body.sum()) * 0.12, 3, 60))
    for _ in range(n_specks):
        idx = int(rng.integers(len(xs)))
        cy, cx = int(ys[idx]), int(xs[idx])
        r = int(rng.integers(1, max(2, min(h, w) // 35)))
        spk = (np.array(SOIL_TONES[int(rng.integers(len(SOIL_TONES)))], np.float32)
               * rng.uniform(0.4, 0.8))
        cv2.circle(stained, (cx, cy), r, spk.tolist(), -1)

    # 5. Faint wet sheen — sparse soft highlights
    sheen = (rng.random((h, w)).astype(np.float32) > 0.990).astype(np.float32)
    if sheen.any():
        sheen = cv2.GaussianBlur(sheen, (0, 0), 1.5) * 55.0
        stained += sheen[..., None]

    # Apply only inside the body; leave transparent pixels untouched
    out = arr.copy()
    out[..., :3] = np.where(body[..., None], np.clip(stained, 0, 255), rgb)
    return Image.fromarray(out.astype(np.uint8), "RGBA")


def scatter_soil_debris(tray_rgb_np, center_xy, radius, n_specks=40, rng=None):
    """Sprinkle small dark soil specks around a point (in place). These are
    UNLABELLED background clutter — they teach the detector to ignore soil
    clumps that superficially resemble small insects (fewer false positives)."""
    rng = rng if rng is not None else np.random.default_rng()
    H, W = tray_rgb_np.shape[:2]
    cx, cy = center_xy
    for _ in range(int(n_specks)):
        ang = rng.uniform(0, 2 * np.pi)
        rr  = abs(rng.normal(0, radius))
        px  = int(np.clip(cx + rr * np.cos(ang), 0, W - 1))
        py  = int(np.clip(cy + rr * np.sin(ang), 0, H - 1))
        sr  = int(rng.integers(1, 5))
        tone = (np.array(SOIL_TONES[int(rng.integers(len(SOIL_TONES)))], np.float32)
                * rng.uniform(0.45, 1.0)).astype(int).tolist()
        cv2.circle(tray_rgb_np, (px, py), sr, tone, -1)


def add_mud_stain(tray_rgb_np, center_xy, radius, rng=None):
    """Overlay a soft, irregular translucent tan stain (soil-stained water)
    on the tray (in place). Operates on a local crop for speed."""
    rng = rng if rng is not None else np.random.default_rng()
    H, W = tray_rgb_np.shape[:2]
    cx, cy = center_xy
    R = int(radius * 2.2)
    x1, y1 = max(0, cx - R), max(0, cy - R)
    x2, y2 = min(W, cx + R), min(H, cy + R)
    if x2 <= x1 or y2 <= y1:
        return
    crop = tray_rgb_np[y1:y2, x1:x2].astype(np.float32)
    ch, cw = crop.shape[:2]
    lcx, lcy = cx - x1, cy - y1
    yy, xx = np.mgrid[0:ch, 0:cw]
    d = np.sqrt((xx - lcx) ** 2 + (yy - lcy) ** 2).astype(np.float32)
    base  = np.clip(1.0 - d / (radius * rng.uniform(1.0, 1.6)), 0, 1)
    noise = cv2.GaussianBlur(rng.standard_normal((ch, cw)).astype(np.float32),
                             (0, 0), max(1.0, radius * 0.3))
    mask = np.clip(base + noise * 0.30, 0, 1)
    mask = cv2.GaussianBlur(mask, (0, 0), max(1.0, radius * 0.12))
    tone = np.array(MUD_STAIN_TONE, np.float32)
    a = (mask * rng.uniform(0.12, 0.35))[..., None]
    tray_rgb_np[y1:y2, x1:x2] = np.clip((1 - a) * crop + a * tone, 0, 255).astype(np.uint8)


print("Soil/brown augmentation toolkit loaded "
      "(apply_soil_staining, scatter_soil_debris, add_mud_stain).")


# ============================================================
# TEXTURE-BASED SOIL BED  (calibrated to REAL pitfall trays)
# ------------------------------------------------------------
# Real Group5 (soiled) trays measured: brightness ~100-150, ~40-52% dark pixels,
# warm cast (R-B ~20-30), tan/brown RGB ~ (122,112,93). The soil is a CONTINUOUS
# fine-grained sediment BED, organically clustered into patches with clean tan
# zones between -- NOT discrete round blobs. apply_soil() renders exactly that.
# (Replaces the old apply_global_soil_wash / scatter_soil_clumps /
# scatter_fine_specks / bury_insects toolkit, which drew artificial polka-dots.)
# ============================================================

# per-tray soil intensity; the brown subset leans SOILED (clean trays cover the
# clean end). 0.5 -> moderate (bright ~130), 1.0 -> heavy grainy bed (bright ~94,
# matching the real heavy pitfall trays).
BROWN_SOIL_LEVEL_RANGE = (0.50, 1.00)

# sediment tones sampled from the real soiled trays (RGB)
_SOIL_TAN   = np.array([158, 140, 112], np.float32)   # thin wash / sediment
_SOIL_BROWN = np.array([112, 90,  62],  np.float32)   # medium sediment
_SOIL_DARK  = np.array([72,  57,  40],  np.float32)   # thick soil / stain core (not near-black)


def _soil_lowfreq(H, W, rng, feat_px):
    """Smooth value-noise field in [0,1], feature size ~feat_px."""
    sh, sw = max(2, int(H / feat_px)), max(2, int(W / feat_px))
    n = rng.random((sh, sw)).astype(np.float32)
    return np.clip(cv2.resize(n, (W, H), interpolation=cv2.INTER_CUBIC), 0, 1)


def _soil_norm(a):
    a = a - a.min()
    return a / (a.max() + 1e-6)


def apply_soil(tray_rgb_np, rng=None, soil_level=0.6):
    """Render a realistic soil BED onto the tray IN PLACE (uint8 RGB, HxWx3).

    Fine-grained sediment + soft crumb aggregates + diffuse stains, clustered
    into organic patches with clean tan zones between, darkened/warmed to match
    real soiled trays. `soil_level` in [0,1] scales coverage & darkness."""
    rng = rng if rng is not None else np.random.default_rng()
    H, W = tray_rgb_np.shape[:2]
    out = tray_rgb_np.astype(np.float32)
    md = max(H, W)

    # 1. density field: 1-2 dominant soiled regions + detail, HIGH contrast so
    #    large tan areas stay CLEAN (as in real trays).
    dens = _soil_norm(0.55 * _soil_lowfreq(H, W, rng, md * 0.33)
                      + 0.30 * _soil_lowfreq(H, W, rng, md * 0.11)
                      + 0.15 * _soil_lowfreq(H, W, rng, md * 0.04))
    gamma = 2.6 - 1.9 * soil_level
    amt = np.clip(dens ** gamma * (0.42 + 0.85 * soil_level), 0, 1)
    amt = cv2.GaussianBlur(amt, (0, 0), md * 0.006)

    # 2. sediment base: tan -> brown ONLY (keep base tan-brown; GRAIN darkens it)
    a2 = amt[..., None]
    sediment = _SOIL_TAN + (_SOIL_BROWN - _SOIL_TAN) * a2
    cover = np.clip((amt - 0.05) / 0.95, 0, 1)[..., None]   # <0.05 => stays clean tray
    out = out * (1 - cover) + sediment * cover

    # 3. WEAK large-scale tone variation only -- grain (below), NOT this, is the
    #    darkener (a strong smooth wash looked like out-of-focus smudges).
    stain = _soil_norm(_soil_lowfreq(H, W, rng, md * 0.05))
    stain = np.clip((stain - 0.55) / 0.45, 0, 1) * np.clip((amt - 0.3) / 0.7, 0, 1)
    stain = cv2.GaussianBlur(stain, (0, 0), md * 0.005)
    sc = (stain * 0.18)[..., None]
    out = out * (1 - sc) + _SOIL_DARK * sc

    # 4. DENSE multi-scale GRAIN = the main texture AND darkener (grainy, not
    #    smooth). Clumped + gated to soiled zones; density scales with soil so
    #    heavy trays become a dense granular bed (matches real heavy pitfall trays).
    clump = _soil_lowfreq(H, W, rng, md * 0.03)
    gate = np.clip((amt - 0.12) / 0.88, 0, 1) * (0.55 + 0.45 * clump)
    g = rng.random((int(H * 0.85), int(W * 0.85))).astype(np.float32)
    g = cv2.resize(g, (W, H), interpolation=cv2.INTER_NEAREST)
    speck = g > (1.0 - 0.62 * gate)
    out[speck] = out[speck] * 0.45 + (_SOIL_BROWN * rng.uniform(0.5, 0.95)) * 0.55
    g2 = rng.random((int(H * 0.5), int(W * 0.5))).astype(np.float32)
    g2 = cv2.resize(g2, (W, H), interpolation=cv2.INTER_NEAREST)
    speck2 = g2 > (1.0 - 0.30 * gate)
    out[speck2] = out[speck2] * 0.35 + _SOIL_DARK * 0.65
    g3 = rng.random((int(H * 0.95), int(W * 0.95))).astype(np.float32)
    g3 = cv2.resize(g3, (W, H), interpolation=cv2.INTER_NEAREST)
    speck3 = g3 > (1.0 - 0.18 * gate)
    out[speck3] = _SOIL_DARK * rng.uniform(0.7, 1.0)

    # 5. a FEW soft crumb aggregates (soft-edged, not hard circles)
    n_crumb = int(rng.integers(6, 16) * (0.4 + soil_level))
    ys, xs = np.where(amt > 0.5)
    if len(xs):
        crumb = np.zeros((H, W), np.float32)
        for _ in range(n_crumb):
            k = int(rng.integers(len(xs)))
            cy, cx = int(ys[k]), int(xs[k])
            rad = int(rng.integers(max(3, H // 220), max(6, H // 90)))
            y0, y1c = max(0, cy - rad), min(H, cy + rad)
            x0, x1c = max(0, cx - rad), min(W, cx + rad)
            yy, xx = np.mgrid[y0:y1c, x0:x1c]
            blob = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (rad * 0.55) ** 2)))
            crumb[y0:y1c, x0:x1c] = np.maximum(crumb[y0:y1c, x0:x1c], blob)
        crumb = cv2.GaussianBlur(crumb, (0, 0), 1.5)
        cc = (crumb * rng.uniform(0.5, 0.75))[..., None]
        out = out * (1 - cc) + _SOIL_DARK * 0.8 * cc

    tray_rgb_np[...] = np.clip(out, 0, 255).astype(np.uint8)


print("Texture-based soil bed loaded (apply_soil -- realistic fine-grained "
      "sediment calibrated to real pitfall trays).")


In [ ]:
def generate_synthetic_tray_brown(
    tray_path,
    insect_meta_pool,
    num_insects,
    cluster_size=(2, 5),
    stain_rng=None,
    soil_level=None,
):
    """Generate a soil-themed tray that matches REAL soiled-trap TEXTURE.

    Real pitfall trays = a continuous fine-grained tan/brown SEDIMENT BED with
    soft, organically-clustered density (brightness ~100-150, warm cast), NOT
    discrete round blobs. Pipeline:
      1. Render a realistic soil BED on the tray (apply_soil) at per-tray
         `soil_level` -- fine grain + soft crumb aggregates + diffuse stains,
         clustered into patches with clean tan zones between (calibrated to the
         real Group5 trays).
      2. Place insects ON TOP of the bed (fully VISIBLE), grouped into
         touching / overlapping clusters, LIGHTLY soil-stained for wet realism.
      3. NO burial / NO over-insect clutter -- keeps insect outlines intact
         (aggressive burial previously taught a size-based catch-all on real data).

    `soil_level` (0..1) sets per-tray intensity; default random in
    BROWN_SOIL_LEVEL_RANGE so a batch spans light -> heavily soiled trays.

    Returns (tray_bgr, placement_log) -- same schema as generate_synthetic_tray,
    so the COCO export consumes it unchanged.
    """
    rng = stain_rng if stain_rng is not None else np.random.default_rng()
    if soil_level is None:
        soil_level = float(rng.uniform(*BROWN_SOIL_LEVEL_RANGE))

    tray_pil = load_tray(tray_path)
    tw, th = tray_pil.size
    margin_x = int(tw * PLACEMENT_MARGIN)
    margin_y = int(th * PLACEMENT_MARGIN)
    placement_log = []

    # ── 1. Realistic soil BED (rendered UNDER the insects) ──
    tray_rgb = np.array(tray_pil.convert("RGB"))
    apply_soil(tray_rgb, rng=rng, soil_level=soil_level)
    tray_pil = Image.fromarray(tray_rgb, "RGB").convert("RGBA")

    # ── 2. Place insects ON TOP, in touching / overlapping clusters ──
    # `placed` only advances on a SUCCESSFUL paste, so a pool whose cutouts all fail
    # to load (moved/deleted files) would spin here forever on a headless server.
    placed = 0
    _tries = 0
    _budget = 20 * max(num_insects, 1)
    while placed < num_insects:
        _tries += 1
        if _tries > _budget:
            print(f"  WARNING: only placed {placed}/{num_insects} after {_budget} "
                  f"attempts — cutouts in this pool are failing to load. Tray kept "
                  f"with what it has.")
            break
        n_in_cluster = min(int(rng.integers(cluster_size[0], cluster_size[1] + 1)),
                           num_insects - placed)
        ccx = int(rng.integers(margin_x, max(margin_x + 1, tw - margin_x)))
        ccy = int(rng.integers(margin_y, max(margin_y + 1, th - margin_y)))
        cluster_spread = int(min(tw, th) * rng.uniform(0.015, 0.05))

        for _k in range(n_in_cluster):
            insect_data = random.choice(insect_meta_pool)
            insect_pil = load_cutout(insect_data)
            if insect_pil is None:
                continue

            insect_np = np.array(insect_pil)
            target_h, target_w = calculate_insect_target_size(
                insect_np, insect_data['dish_radius'], (th, tw),
                insect_data['scale_factor'], insect_data['area_fraction'],
            )
            del insect_np
            if target_h < 2 or target_w < 2:
                insect_pil.close()
                continue

            insect_pil = insect_pil.resize((target_w, target_h), Image.Resampling.LANCZOS)

            # LIGHT wet/soil staining -- keep specimens clearly VISIBLE on top
            insect_pil = apply_soil_staining(
                insect_pil, intensity=float(rng.uniform(0.12, 0.35)), rng=rng)

            angle = random.uniform(*ROTATION_RANGE)
            if abs(angle) > 0.5:
                insect_pil = insect_pil.rotate(
                    angle, resample=Image.Resampling.BICUBIC, expand=True)

            iw, ih = insect_pil.size      # canvas: used for PLACEMENT only

            # Same correction as the clean-tray generator: the recorded box must be
            # the insect, not the rotated canvas whose corners are transparent.
            # Brown trays are ~35% of the dataset, so leaving this unfixed would give
            # two different box conventions in one dataset.
            _ab = insect_pil.split()[-1].point(lambda v: 255 if v > 10 else 0).getbbox()
            if _ab is None:
                insect_pil.close()
                continue
            _bx0, _by0, _bx1, _by1 = _ab

            # place near the cluster centre with gaussian jitter (allows touching)
            x = int(ccx + rng.normal(0, cluster_spread) - iw / 2)
            y = int(ccy + rng.normal(0, cluster_spread) - ih / 2)
            x = max(margin_x, min(x, max(margin_x, tw - iw - margin_x)))
            y = max(margin_y, min(y, max(margin_y, th - ih - margin_y)))

            tray_pil = place_insect_on_tray_pil(tray_pil, insect_pil, x, y)

            placement_log.append({
                'processid':     insect_data['processid'],
                'class':         insect_data.get('class', 'Unknown'),
                'order':         insect_data.get('order', 'Unknown'),
                # the insect's true extent, offset into tray coordinates
                'x': x + _bx0, 'y': y + _by0,
                'width': _bx1 - _bx0, 'height': _by1 - _by0,
                'x_canvas': x, 'y_canvas': y, 'width_canvas': iw, 'height_canvas': ih,
                'angle':         angle,
                'scale_factor':  insect_data['scale_factor'],
                'area_fraction': insect_data['area_fraction'],
            })
            insect_pil.close()
            placed += 1
            if placed >= num_insects:
                break

    # NO burial / NO over-insect clutter -- insects stay fully visible on the bed.
    tray_bgr = cv2.cvtColor(np.array(tray_pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    tray_pil.close()
    return tray_bgr, placement_log


print("Brown tray generator defined "
      "(generate_synthetic_tray_brown -- realistic soil bed, insects on top, no burial).")


---
### Cell 17b — TEST: double-check brown generation

Run this first. Shows original-vs-stained cutouts, one full sample tray, and zoomed clusters. Re-run for a new random sample.


In [ ]:
# ============================================================
# TEST — Brown / soil-stained generation (visual sanity check)
# ============================================================
# Run this BEFORE the full batch cell to confirm:
#   (1) staining looks right on individual cutouts
#   (2) clusters / pairs form and look natural
#   (3) debris + mud stains read as "soil trap"
# Re-run for a fresh random sample.
import matplotlib.pyplot as plt

# Source pool of cutouts — AUTO-LOADS from disk if Cell 12 was not run,
# so you can test the brown set without generating the clean trays.
try:
    _test_pool = all_meta_filtered           # quality-filtered pool (Cell 12)
except NameError:
    try:
        _test_pool = all_meta                # raw pool (Cell 10/12, in memory)
    except NameError:
        _cut_csv = OUTPUT_DIR / "cutouts" / "cutouts_metadata.csv"
        all_meta = pd.read_csv(_cut_csv).to_dict("records")
        all_meta = [m for m in all_meta if Path(m["cutout_path"]).exists()]
        _test_pool = all_meta
        print(f"Loaded {len(all_meta)} cached cutouts from {_cut_csv}")
print(f"Test pool: {len(_test_pool)} cutouts")

_trng = np.random.default_rng()

# ── 1. Before / after staining on a few cutouts ──
_samples = random.sample(_test_pool, min(6, len(_test_pool)))
_n = len(_samples)
fig, axes = plt.subplots(2, _n, figsize=(3.2 * _n, 7))
if _n == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for _j, _m in enumerate(_samples):
    _orig = load_cutout(_m)
    if _orig is None:
        continue
    _stained = apply_soil_staining(_orig.copy(), rng=_trng)
    axes[0][_j].imshow(_orig)
    axes[0][_j].set_title(f"{_m.get('order', '?')}\noriginal", fontsize=9)
    axes[0][_j].axis("off")
    axes[1][_j].imshow(_stained)
    axes[1][_j].set_title("soil-stained", fontsize=9)
    axes[1][_j].axis("off")
plt.suptitle("Soil staining: original (top) vs brown-stained (bottom)", fontsize=13)
plt.tight_layout()
plt.show()

# ── 2. One full sample brown tray (clustered) ──
_tp   = random.choice(sorted(TRAY_DIR.rglob("*.png")))
_pool = random.sample(_test_pool, min(len(_test_pool), 120))
_btray, _bplog = generate_synthetic_tray_brown(
    _tp, _pool, random.randint(*INSECTS_PER_TRAY), stain_rng=_trng)

plt.figure(figsize=(13, 10))
plt.imshow(cv2.cvtColor(_btray, cv2.COLOR_BGR2RGB))
plt.title(f"Sample brown tray — {len(_bplog)} insects in clusters "
          f"(bg: {_tp.name})", fontsize=12)
plt.axis("off")
plt.show()

# Optional: save it so you can open it at full resolution
_test_out = OUTPUT_DIR / "TEST_brown_tray.png"
cv2.imwrite(str(_test_out), _btray)
print(f"Saved full-res sample → {_test_out}")

# ── 3. Zoom into the first cluster to inspect blending / overlap ──
if _bplog:
    _df = pd.DataFrame(_bplog)
    _k = min(5, len(_df))
    fig, axes = plt.subplots(1, _k, figsize=(3.4 * _k, 3.6))
    if _k == 1:
        axes = [axes]
    for _i in range(_k):
        _r = _df.iloc[_i]
        _x, _y = int(_r["x"]), int(_r["y"])
        _w, _h = int(_r["width"]), int(_r["height"])
        _pad = max(_w, _h)
        _crop = _btray[max(0, _y - _pad):_y + _h + _pad,
                       max(0, _x - _pad):_x + _w + _pad]
        axes[_i].imshow(cv2.cvtColor(_crop, cv2.COLOR_BGR2RGB))
        axes[_i].set_title(f"{_r.get('order', '?')}\n{_w}x{_h}px", fontsize=9)
        axes[_i].axis("off")
    plt.suptitle("Zoom: brown-stained clustered insects", fontsize=12)
    plt.tight_layout()
    plt.show()


---
### Cell 17c — Batch: generate the brown subset


In [ ]:
# ============================================================
# BATCH — Generate the brown / soil-stained tray subset
# ============================================================
# Writes brown_tray_<split>_####.png + matching _placements.csv into OUTPUT_DIR.
# The brown subset is generated PER SPLIT from the SAME specimen-disjoint pools
# built in the clean-tray cell (split_pools), so a brown train tray only ever
# contains train specimens — no leakage into valid/test. The COCO export picks
# up both prefixes by their split tag.
from tqdm.auto import tqdm
import gc

# Brown subset sized as a FRACTION of the clean-tray count, so the model
# sees plenty of soiled trays (real deployment includes soiled traps).
BROWN_FRACTION_OF_CLEAN = 0.35     # brown trays as a fraction of clean trays
try:
    _n_clean = int(NUM_TRAYS_TO_GENERATE)            # from the config cell
except NameError:
    _n_clean = 2800                                  # fallback if cell not run
NUM_BROWN_TRAYS   = int(round(_n_clean * BROWN_FRACTION_OF_CLEAN))  # bumped (was 600)
BROWN_PREFIX      = "brown_tray"
CLUSTER_SIZE      = (2, 5)       # insects per cluster (2 = pairs)
BROWN_SPLIT_FRACS = {'train': 0.60, 'valid': 0.20, 'test': 0.20}

# Requires the specimen-level disjoint split from the clean-tray cell.
try:
    split_pools
except NameError:
    raise RuntimeError(
        "split_pools not found — run the clean-tray cell (Cell 27) first; "
        "it builds the specimen-level disjoint split this cell draws from."
    )

_brown_rng = np.random.default_rng(RANDOM_SEED if RANDOM_SEED is not None else 42)
_tray_bgs  = sorted(TRAY_DIR.rglob("*.png"))
print(f"Tray backgrounds: {len(_tray_bgs)}")
print(f"Generating {NUM_BROWN_TRAYS} brown trays "
      f"({INSECTS_PER_TRAY[0]}-{INSECTS_PER_TRAY[1]} insects, clusters of "
      f"{CLUSTER_SIZE[0]}-{CLUSTER_SIZE[1]})")
print("=" * 60)

_brown_logs     = []
_brown_tally    = {}
_brown_by_split = {}

# The brown subset now uses the SAME balanced schedule as the clean trays.
# It used to draw every tray from one flat pool of the split's specimens, which is
# frequency-weighted: Coleoptera holds thousands of specimens and Zoraptera six, so a
# rare order reached a brown tray roughly never. That left 26% of the dataset doing
# nothing for exactly the classes that score worst. Reusing _build_schedule and
# _build_pool gives brown trays the same guarantees the clean ones get: mono trays per
# rich class, one tray per rich pair, and two spotlight trays per non-rich class where
# that order is 35% of the insects present.
_brown_sysmin = {}

for _split in ('train', 'valid', 'test'):
    if not split_pools[_split]:
        continue
    # Rebind the globals _build_pool and the recipes read, to THIS split's pools —
    # exactly as the clean-tray cell does, so the brown subset stays specimen-disjoint.
    class_pools   = split_pools[_split]
    all_orders    = list(class_pools.keys())
    RICH_ORDERS   = [o for o in all_orders if len(class_pools[o]) >= 200]
    MEDIUM_ORDERS = [o for o in all_orders if 50 <= len(class_pools[o]) < 200]
    SPARSE_ORDERS = [o for o in all_orders if 10 <= len(class_pools[o]) < 50]
    RARE_ORDERS   = [o for o in all_orders if 0 < len(class_pools[o]) < 10]

    _n_brown = int(round(NUM_BROWN_TRAYS * BROWN_SPLIT_FRACS[_split]))
    _bschedule, _bsysmin = _build_schedule(_n_brown)
    _brown_sysmin[_split] = _bsysmin
    if _n_brown < _bsysmin:
        print(f"  note: [{_split}] {_n_brown} brown trays is below the systematic "
              f"minimum of {_bsysmin}, so not every class and pair gets its tray")
    print(f"[{_split}] {len(_bschedule)} brown trays (systematic min {_bsysmin}) | "
          f"{len(all_orders)} orders | "
          f"{sum(len(v) for v in class_pools.values())} specimens")

    for _i, _bfracs in enumerate(tqdm(_bschedule, desc=f"Brown {_split}")):
        _tpath = random.choice(_tray_bgs)
        _n_ins = random.randint(*INSECTS_PER_TRAY)
        # class-balanced pool for THIS tray, from THIS split's specimens only
        _pool  = _build_pool(_bfracs)

        _tray_bgr, _plog = generate_synthetic_tray_brown(
            _tpath, _pool, _n_ins, cluster_size=CLUSTER_SIZE, stain_rng=_brown_rng)

        _out_name = f"{BROWN_PREFIX}_{_split}_{_i + 1:04d}{TRAY_EXT}"
        cv2.imwrite(str(OUTPUT_DIR / _out_name), _tray_bgr, _TRAY_PARAMS)
        pd.DataFrame(_plog).to_csv(
            OUTPUT_DIR / f"{BROWN_PREFIX}_{_split}_{_i + 1:04d}_placements.csv",
            index=False)
        del _tray_bgr

        for _e in _plog:
            _o = _e.get('order', 'Unknown')
            _brown_tally[_o] = _brown_tally.get(_o, 0) + 1
            _brown_by_split.setdefault(_split, {})
            _brown_by_split[_split][_o] = _brown_by_split[_split].get(_o, 0) + 1

        _brown_logs.append({
            'split':       _split,
            'tray_index':  _i + 1,
            'background':  _tpath.name,
            'num_insects': len(_plog),
            'output_path': str(OUTPUT_DIR / _out_name),
        })
        if (_i + 1) % 25 == 0:
            gc.collect()

print("\n" + "=" * 60)
print(f"Brown subset complete: {len(_brown_logs)} trays -> {OUTPUT_DIR}")
print("\nBrown-subset class distribution — insect share vs the share of")
print("SPECIMENS the order holds. Balancing shows up as the left column")
print("being larger than the right for the rare orders.\n")
_tot_b = sum(_brown_tally.values()) or 1
_spec_n = {}
for _sp in split_pools:
    for _o, _ms in split_pools[_sp].items():
        _spec_n[_o] = _spec_n.get(_o, 0) + len(_ms)
_tot_s = sum(_spec_n.values()) or 1
print(f"  {'order':24s}{'insects':>9s}{'of brown':>10s}{'of specimens':>14s}   ")
for _o, _c in sorted(_brown_tally.items(), key=lambda kv: -kv[1]):
    if not _c:
        continue
    _bs = _c / _tot_b
    _ss = _spec_n.get(_o, 0) / _tot_s
    _tag = "boosted" if _bs > 2 * _ss else ("thin" if _bs < 0.002 else "")
    print(f"  {_o:24s}{_c:>9d}{_bs:>10.2%}{_ss:>14.2%}   {_tag}")

_absent = [o for o in _spec_n if not _brown_tally.get(o)]
if _absent:
    print(f"\n  WARNING: no brown tray contains {sorted(_absent)}")
print(f"\nper split: " + ", ".join(
    f"{_sp} {len(_brown_by_split.get(_sp, {}))} orders" for _sp in ('train','valid','test')))


---
## Cell 16 — Export Dataset to COCO Format (for RF-DETR)

RF-DETR expects Roboflow's COCO layout:
```
insect_dataset_coco/
  train/
    _annotations.coco.json   ← COCO JSON (images + annotations + categories)
    synthetic_tray_0001.png
    synthetic_tray_0002.png
    ...
  valid/
    _annotations.coco.json
    ...
```

**COCO bbox format:** `[x_min, y_min, width, height]` in **pixel coordinates** (not normalised).

`category_id` starts at 1 (COCO convention). RF-DETR remaps sparse IDs → 0-based labels automatically via `remap_category_ids=True`.


In [ ]:
# ============================================================
# COCO DATASET EXPORT SETTINGS
# ============================================================
# NOTE: the train/valid/test split is now decided at GENERATION time
# (specimen-level disjoint pools, see Cell 27). Trays carry their split in the
# filename — synthetic_tray_<split>_####.png / brown_tray_<split>_####.png — so
# this export simply routes each tray to its pre-assigned split. There is no
# random tray-level slicing here anymore (that caused train/test leakage).
# beside the trays, on the same roomy filesystem — this is another ~75 GB
COCO_DATASET_DIR = BASE_DIR / "insect_dataset_coco_20000_mix"

EXPORT_MAX_SIDE     = None   # keep full res. At 1536 this discarded 42% of the
                             # 2644x2108 backgrounds, putting insects at 56-102 px
                             # against 132 px in the real trays. Full res gives
                             # 96-176 px, which matches.
EXPORT_AS_JPEG      = True
EXPORT_JPEG_QUALITY = 90

CLASS_NAMES = [
    # Insecta — 19 orders
    # Raphidioptera (2 specimens, one adult + one larva) and Siphonaptera (6) are
    # excluded: with specimen-disjoint splits a class this small is tested on
    # individuals nothing in training resembles, so both scored ~0 by construction.
    "Lepidoptera", "Psocodea", "Odonata", "Zoraptera",
    "Phasmatodea", "Zygentoma", "Thysanoptera", "Strepsiptera",
    "Archaeognatha", "Dermaptera",
    "Diptera", "Coleoptera", "Hymenoptera", "Hemiptera",
    "Orthoptera", "Trichoptera", "Neuroptera",
    # Arachnida — 2 orders
    "Araneae", "Opiliones",
    # Myriapods at ORDER level. They were previously class-level labels
    # ("Diplopoda", "Chilopoda"), each covering morphologically unrelated orders —
    # one detector head asked to learn several appearances at once, which is why
    # both scored badly (0.19 and 0.34). Julida, Chordeumatida, Polydesmida and
    # Geophilomorpha are removed by the prune blocklist, so what remains is:
    "Polyxenida", "Lithobiomorpha", "Scutigeromorpha",
]

import shutil, json as _json

# Create directory skeleton
for _split in ["train", "valid", "test"]:
    (COCO_DATASET_DIR / _split).mkdir(parents=True, exist_ok=True)

# ── Collect trays by the split they were GENERATED for (specimen-disjoint) ──
# Filenames encode the split, so no leak-prone re-slicing is needed here.
random.seed(RANDOM_SEED if RANDOM_SEED is not None else 42)

_split_trays = {}
for _split in ("train", "valid", "test"):
    _clean = sorted(OUTPUT_DIR.glob(f"synthetic_tray_{_split}_*{TRAY_EXT}"))
    _brown = sorted(OUTPUT_DIR.glob(f"brown_tray_{_split}_*{TRAY_EXT}"))
    _trays = _clean + _brown
    random.shuffle(_trays)
    _split_trays[_split] = _trays
    print(f"  {_split:5s}: {len(_clean)} clean + {len(_brown)} brown = {len(_trays)} trays")

_train_trays = _split_trays["train"]
_val_trays   = _split_trays["valid"]
_test_trays  = _split_trays["test"]

if not (_train_trays or _val_trays or _test_trays):
    raise FileNotFoundError(
        f"No split-tagged tray images found in {OUTPUT_DIR}.\n"
        "Run the clean-tray cell (Cell 27) and brown cell (Cell 40) first — they "
        "now write split-tagged filenames (synthetic_tray_<split>_####.png)."
    )

# Build processid → order/class lookups so existing placement CSVs work too
_processid_cls = df.set_index('processid')['class'].to_dict()
_processid_ord = df.set_index('processid')['order'].to_dict()
_name_to_cat   = {name: idx + 1 for idx, name in enumerate(CLASS_NAMES)}

# ── Taxonomy pre-scan ──────────────────────────────────────────────────────────
# Every placed insect must resolve to one of CLASS_NAMES. Check that against the
# placement CSVs BEFORE writing a single JPEG: re-encoding ~27,000 full-resolution
# trays only to raise at the end would waste the best part of an hour to tell you
# something that is knowable in seconds.
_scan_bad = {}
for _sp, _tl in _split_trays.items():
    for _tp in _tl:
        _lp = OUTPUT_DIR / f"{_tp.stem}_placements.csv"
        if not _lp.exists():
            continue
        try:
            _d = pd.read_csv(_lp, usecols=lambda c: c in ("processid", "order", "class"))
        except Exception:
            continue
        # dict records, not itertuples: 'class' is a keyword, so itertuples renames it
        for _rec in _d.to_dict("records"):
            _o   = str(_rec.get("order", "") or "")
            _c   = str(_rec.get("class", "") or "")
            _pid = str(_rec.get("processid", ""))
            if _o in _name_to_cat or _c in _name_to_cat:
                continue
            if (_processid_ord.get(_pid, "") in _name_to_cat
                    or _processid_cls.get(_pid, "") in _name_to_cat):
                continue
            _scan_bad[_o or _c or _pid] = _scan_bad.get(_o or _c or _pid, 0) + 1
if _scan_bad:
    raise ValueError(
        "These taxa are on the generated trays but are not in CLASS_NAMES, so they "
        "cannot be labelled:\n  "
        + "\n  ".join(f"{k}: {v} boxes" for k, v in
                      sorted(_scan_bad.items(), key=lambda kv: -kv[1]))
        + "\n\nAlmost certainly the prune blocklist did not load when the trays were "
          "generated. Check PRUNE_REVIEW/cutout_blocklist.csv, then re-run the "
          "generation cells. Nothing has been written yet.")
print(f"taxonomy pre-scan: every placed insect resolves to one of the "
      f"{len(CLASS_NAMES)} classes")


def _build_coco_json(tray_list, split_dir):
    categories = [
        {"id": idx + 1, "name": name, "supercategory": "none"}
        for idx, name in enumerate(CLASS_NAMES)
    ]

    images      = []
    annotations = []
    ann_id      = 1
    _unresolved = []          # specimens whose taxon is not in CLASS_NAMES

    for img_id, tray_path in enumerate(tray_list, start=1):
        stem     = tray_path.stem
        log_path = OUTPUT_DIR / f"{stem}_placements.csv"

        if not log_path.exists():
            print(f"  WARNING: placement log missing for {tray_path.name} — skipping")
            continue

        # If the tray is ALREADY a JPEG at the export resolution, copy it instead of
        # decoding and re-encoding. That skips ~1.2 h over 27,000 trays and, more to
        # the point, avoids a second lossy pass: the delivered image keeps the q95 it
        # was written at rather than becoming q90-of-q95. Only the header is read, to
        # get the size.
        if (EXPORT_AS_JPEG and not EXPORT_MAX_SIDE
                and tray_path.suffix.lower() in (".jpg", ".jpeg")):
            with Image.open(tray_path) as _img:
                out_w, out_h = _img.size
            out_name = tray_path.name
            _scale = 1.0
            shutil.copy2(tray_path, split_dir / out_name)
        else:
          with Image.open(tray_path) as _img:
            _img = _img.convert("RGB")
            tray_w, tray_h = _img.size

            if EXPORT_MAX_SIDE and max(tray_w, tray_h) > EXPORT_MAX_SIDE:
                 _scale = EXPORT_MAX_SIDE / max(tray_w, tray_h)
                 out_w, out_h = round(tray_w * _scale), round(tray_h * _scale)
                 _out_img = _img.resize((out_w, out_h), Image.LANCZOS)
            else:
                 _scale = 1.0
                 out_w, out_h = tray_w, tray_h
                 _out_img = _img

            if EXPORT_AS_JPEG:
                 out_name = tray_path.stem + ".jpg"
                 _out_img.save(split_dir / out_name, "JPEG",
                               quality=EXPORT_JPEG_QUALITY, optimize=True)
            else:
                 out_name = tray_path.name
                 _out_img.save(split_dir / out_name)

        images.append({
            "id":        img_id,
            "file_name": out_name,
            "width":     out_w,
            "height":    out_h,
        })

        # A tray whose insects all failed to place writes a CSV with no columns, and
        # pd.read_csv raises on that. Crashing here would throw away a multi-hour
        # generation run at the last step, so skip the tray and say so.
        try:
            _log = pd.read_csv(log_path)
        except pd.errors.EmptyDataError:
            print(f"  WARNING: {tray_path.name} has an empty placement log — "
                  f"exported as a tray with no insects")
            _log = pd.DataFrame(columns=["processid", "x", "y", "width", "height"])
        for _, _row in _log.iterrows():
            x = _row["x"] * _scale
            y = _row["y"] * _scale
            w = _row["width"] * _scale
            h = _row["height"] * _scale

            x = int(max(0, min(x, out_w - 1)))
            y = int(max(0, min(y, out_h - 1)))
            w = int(max(1, min(w, out_w - x)))
            h = int(max(1, min(h, out_h - y)))

            # Resolve category: Insecta/Arachnida → order label; Diplopoda/Chilopoda → class label
            _pid = str(_row.get('processid', ''))
            _ord_row = str(_row.get('order', ''))
            _cls_row = str(_row.get('class', ''))
            if _ord_row in _name_to_cat:
                _cat_id = _name_to_cat[_ord_row]
            elif _cls_row in _name_to_cat:
                _cat_id = _name_to_cat[_cls_row]
            else:
                # Fallback: join from selected_classes_original.csv via processid
                _ord = _processid_ord.get(_pid, '')
                _cls = _processid_cls.get(_pid, '')
                if _ord in _name_to_cat:
                    _cat_id = _name_to_cat[_ord]
                elif _cls in _name_to_cat:
                    _cat_id = _name_to_cat[_cls]
                else:
                    # NO silent fallback. This used to assign category 1, which
                    # renamed every Julida / Polydesmida / Geophilomorpha /
                    # Chordeumatida / Raphidioptera / Siphonaptera specimen to
                    # "Lepidoptera" — 21 specimens, pasted thousands of times,
                    # poisoning a real class with animals that are not it. The
                    # blocklist is what keeps them out of the pool; if it did not
                    # load, this is where you find out, not six weeks later.
                    _cat_id = None
                    _unresolved.append((_pid, _ord_row or _ord, _cls_row or _cls))
                    continue

            annotations.append({
                "id":          ann_id,
                "image_id":    img_id,
                "category_id": _cat_id,
                "bbox":        [x, y, w, h],
                "area":        float(w * h),
                "iscrowd":     0,
            })
            ann_id += 1

    if _unresolved:
        import collections as _c
        _tally = _c.Counter(f"{o or '?'} (class {c or '?'})" for _, o, c in _unresolved)
        raise ValueError(
            f"{len(_unresolved)} placed insects have a taxon that is not one of the "
            f"{len(CLASS_NAMES)} CLASS_NAMES, so they cannot be labelled:\n  "
            + "\n  ".join(f"{k}: {v} boxes" for k, v in _tally.most_common())
            + f"\n  e.g. processid {_unresolved[0][0]}\n\n"
            "These specimens should never have reached a tray. Almost certainly the "
            "prune blocklist did not load in Cell 27 — check that "
            "PRUNE_REVIEW/cutout_blocklist.csv exists relative to BASE_DIR, then "
            "re-run Cell 27 and the brown cell before exporting again.")

    coco_json = {
        "info":        {"description": f"Synthetic arthropod tray dataset — "
                                       f"{len(CLASS_NAMES)} order-level classes"},
        "categories":  categories,
        "images":      images,
        "annotations": annotations,
    }
    ann_path = split_dir / "_annotations.coco.json"
    with open(ann_path, "w") as _f:
        _json.dump(coco_json, _f)

    return len(images), len(annotations)


_n_imgs_train, _n_ann_train = _build_coco_json(_train_trays, COCO_DATASET_DIR / "train")
_n_imgs_val,   _n_ann_val   = _build_coco_json(_val_trays,   COCO_DATASET_DIR / "valid")
_n_imgs_test,  _n_ann_test  = _build_coco_json(_test_trays,  COCO_DATASET_DIR / "test")

print(f"\nCOCO dataset written to: {COCO_DATASET_DIR}")
print(f"  train: {_n_imgs_train} images, {_n_ann_train} annotations")
print(f"  valid: {_n_imgs_val}  images, {_n_ann_val} annotations")
print(f"  test:  {_n_imgs_test}  images, {_n_ann_test} annotations")
print(f"  classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"  annotation file: <split>/_annotations.coco.json")
print(f"  NOTE: splits are specimen-DISJOINT (no train/test leakage).")


---
## Verify the export

Re-reads what was written to disk and checks it, trusting nothing still in memory.
The important one is **specimen leakage**: it reads every placement CSV and confirms
no individual insect was pasted into more than one split. The second is **box
tightness** — it compares each recorded box against the rotated canvas it was cut
from, which is the direct end-to-end proof that the canvas-inflation bug is gone.

Run this before copying anything to the training machine.


In [ ]:
# ============================================================
# VERIFY THE EXPORT — run this before copying anything to the training machine
# ============================================================
# Nothing here trusts a variable still in memory: every check re-reads what was
# actually written to disk. If the last line prints, the dataset is sound.
import json as _vj, random as _vr
from collections import Counter as _VC, defaultdict as _VD
import numpy as _vnp
from PIL import Image as _VI

_V = {}
for _sp in ("train", "valid", "test"):
    _f = COCO_DATASET_DIR / _sp / "_annotations.coco.json"
    assert _f.exists(), f"missing {_f} — run the export cell above first"
    _V[_sp] = _vj.load(open(_f))
_VCATS = sorted(_V["train"]["categories"], key=lambda c: c["id"])
_VNAMES = [c["name"] for c in _VCATS]
_vok = []

# 1 ─ ids, categories, references
for _sp, _c in _V.items():
    _ii = [i["id"] for i in _c["images"]]; _aa = [a["id"] for a in _c["annotations"]]
    assert len(_ii) == len(set(_ii)), f"{_sp}: duplicate image ids"
    assert len(_aa) == len(set(_aa)), f"{_sp}: duplicate annotation ids"
    assert not ({a["image_id"] for a in _c["annotations"]} - set(_ii)), \
        f"{_sp}: annotations reference missing images"
    assert [(x["id"], x["name"]) for x in sorted(_c["categories"], key=lambda x: x["id"])] \
           == [(x["id"], x["name"]) for x in _VCATS], f"{_sp}: category list differs"
    assert not ({a["category_id"] for a in _c["annotations"]} - {x["id"] for x in _VCATS}), \
        f"{_sp}: annotation uses an undeclared category id"
_vok.append(f"ids unique, no orphans, all three splits declare the same {len(_VCATS)} classes")

# 2 ─ files on disk, and the recorded size is the true size
for _sp, _c in _V.items():
    _miss = [i["file_name"] for i in _c["images"]
             if not (COCO_DATASET_DIR / _sp / i["file_name"]).exists()]
    assert not _miss, f"{_sp}: {len(_miss)} image files missing, e.g. {_miss[:3]}"
    for _i in _vr.Random(0).sample(_c["images"], min(40, len(_c["images"]))):
        with _VI.open(COCO_DATASET_DIR / _sp / _i["file_name"]) as _im: _w, _h = _im.size
        assert (_w, _h) == (_i["width"], _i["height"]), \
            f"{_sp}/{_i['file_name']}: json {_i['width']}x{_i['height']} vs file {_w}x{_h}"
_vok.append("every image file exists; sampled sizes match the JSON")

# 3 ─ boxes inside the image, with real area
for _sp, _c in _V.items():
    _dim = {i["id"]: (i["width"], i["height"]) for i in _c["images"]}
    _bad = [a for a in _c["annotations"]
            if a["bbox"][2] <= 0 or a["bbox"][3] <= 0
            or a["bbox"][0] < 0 or a["bbox"][1] < 0
            or a["bbox"][0]+a["bbox"][2] > _dim[a["image_id"]][0]+1
            or a["bbox"][1]+a["bbox"][3] > _dim[a["image_id"]][1]+1]
    assert not _bad, f"{_sp}: {len(_bad)} degenerate/out-of-bounds boxes, e.g. {_bad[0]}"
_vok.append("every box has positive area and lies inside its image")

# 4 ─ LEAKAGE: no SPECIMEN may appear in two splits
_pid_split = _VD(set)
_spec = _VD(set)          # (class, split) -> distinct specimens actually placed
for _sp in ("train", "valid", "test"):
    for _pat in ("synthetic_tray", "brown_tray"):
        for _csv in OUTPUT_DIR.glob(f"{_pat}_{_sp}_*_placements.csv"):
            try: _d = pd.read_csv(_csv, usecols=lambda c: c in ("processid", "order"))
            except Exception: continue
            for _r in _d.to_dict("records"):
                _p = str(_r.get("processid", ""))
                _pid_split[_p].add(_sp)
                _spec[(str(_r.get("order", "?")), _sp)].add(_p)
_span = {p: sorted(v) for p, v in _pid_split.items() if len(v) > 1}
assert not _span, (f"SPECIMEN LEAKAGE — {len(_span)} individuals appear in more than "
                   f"one split, e.g. {dict(list(_span.items())[:5])}")
_vok.append(f"{len(_pid_split)} specimens used, each confined to exactly one split")

# 4b ─ BLOCKLIST: prove it held, by looking at what was actually pasted.
# The pool filter in the generation cell is the intent; this is the evidence. It also
# catches the case the export's taxonomy pre-scan cannot see — a blocked specimen whose
# order IS a valid class (the four pruned Odonata fragments), which would otherwise
# come back silently if the blocklist file were truncated or half-written.
_blp = BASE_DIR / "PRUNE_REVIEW" / "cutout_blocklist.csv"
if _blp.exists():
    _blk = pd.read_csv(_blp)
    _blset = set(_blk["processid"].astype(str))
    _leaked = sorted(_blset & set(_pid_split))
    assert not _leaked, (
        f"{len(_leaked)} BLOCKLISTED specimens were pasted onto trays anyway: "
        f"{_leaked[:8]}\n"
        "The blocklist did not take effect when the trays were generated. Re-run the "
        "generation cells with it in place — the trays on disk are not usable.")
    _vok.append(f"none of the {len(_blset)} blocklisted specimens reached a tray "
                f"(checked against every placement CSV)")
    if "reason" in _blk.columns:
        print("blocklist held. Excluded, by reason:")
        for _r, _n in _blk["reason"].value_counts().items():
            print(f"   {_n:3d}  {_r}")
else:
    print(f"NOTE: no blocklist at {_blp} to check against — if the trays were generated "
          f"without one, the pruned specimens are in this dataset.")

# 5 ─ no tray file in two splits
_seen = _VD(set)
for _sp, _c in _V.items():
    for _i in _c["images"]: _seen[_i["file_name"]].add(_sp)
assert not {f for f, v in _seen.items() if len(v) > 1}, "a tray file is in two splits"
_vok.append("no tray appears in more than one split")

# 6 ─ THE BOX ITSELF: does the recorded box match the visible insect?
# The rotated-canvas bug inflated boxes by ~1.29x. Re-derive the insect's extent
# from the placement CSV's canvas columns and confirm the exported box is the
# TIGHT one, then look at how much of each box is actually insect.
_fill = []
for _sp in ("train", "valid", "test"):
    _c = _V[_sp]
    for _i in _vr.Random(1).sample(_c["images"], min(12, len(_c["images"]))):
        _stem = _i["file_name"].rsplit(".", 1)[0]
        _lp = OUTPUT_DIR / f"{_stem}_placements.csv"
        if not _lp.exists(): continue
        _d = pd.read_csv(_lp)
        if not {"width_canvas", "height_canvas"} <= set(_d.columns): continue
        _r = (_d["width"] * _d["height"]) / (_d["width_canvas"] * _d["height_canvas"])
        _fill += _r.dropna().tolist()
if _fill:
    _fill = _vnp.array(_fill)
    _ratio = 1.0 / _vnp.sqrt(_fill.mean())
    print(f"box tightness: the recorded box covers {_fill.mean():6.1%} of the rotated "
          f"canvas\n              i.e. the canvas box would have been "
          f"{_ratio:.3f}x too large on a side")
    assert _fill.mean() < 0.999, (
        "the exported boxes ARE the canvas boxes — the rotation fix is not in effect")
    _vok.append(f"boxes are the insect, not the rotated canvas "
                f"({_ratio:.2f}x tighter per side)")
else:
    print("box tightness: no placement CSV carried the canvas columns — skipped")

# 7 ─ composition and scale
_rows = []
for _sp, _c in _V.items():
    _o = {i["id"]: ("brown" if i["file_name"].startswith("brown_tray") else "clean")
          for i in _c["images"]}
    for _a in _c["annotations"]:
        _rows.append((_sp, _o[_a["image_id"]],
                      _VNAMES[[x["id"] for x in _VCATS].index(_a["category_id"])],
                      _vnp.sqrt(_a["bbox"][2] * _a["bbox"][3])))
_DF = pd.DataFrame(_rows, columns=["split", "tray", "class", "px"])

print("\ncomposition\n")
_t = []
for _sp, _c in _V.items():
    _nc = sum(1 for i in _c["images"] if not i["file_name"].startswith("brown_tray"))
    _nb = len(_c["images"]) - _nc
    _t.append({"split": _sp, "clean": _nc, "brown": _nb, "trays": len(_c["images"]),
               "insects": len(_c["annotations"]),
               "brown share": f"{_nb/max(len(_c['images']),1):.1%}"})
print(pd.DataFrame(_t).to_string(index=False))

print(f"\ninsect size across the whole export: median "
      f"{_DF.px.median():.0f} px, 5-95% {_DF.px.quantile(.05):.0f}-{_DF.px.quantile(.95):.0f} px")
print(f"   real 3DTRAB insects measure ~148 px on the same footing, so the two sit "
      f"{148/max(_DF.px.median(),1):.2f}x apart")

print("\nper-class boxes (a class missing from a split cannot be scored there)\n")
_p = _DF.pivot_table(index="class", columns="split", values="px",
                     aggfunc="count", fill_value=0)
_p = _p.reindex(columns=["train", "valid", "test"], fill_value=0)
_p["total"] = _p.sum(1)
print(_p.sort_values("total", ascending=False).to_string())
_gap = [(c, s) for c in _p.index for s in ("train", "valid", "test") if _p.loc[c, s] == 0]
_missing = sorted(set(_VNAMES) - set(_p.index))
if _gap:    print(f"\nWARNING: class absent from a split: {_gap}")
if _missing: print(f"WARNING: declared but never placed: {_missing}")
assert not _missing, ("a declared class was never placed on any tray — its specimens "
                      "were probably all removed by the blocklist, so it must come "
                      "out of CLASS_NAMES too")

# ── how many DISTINCT individuals is each class actually built from? ──
# Boxes are cheap: one specimen pasted 4,000 times is still one animal. A class
# tested on two individuals nothing in training resembles will score near zero no
# matter how many boxes it has, and this table is what says so.
print("\n\ndistinct SPECIMENS behind each class (boxes in brackets)\n")
_cls_all = sorted({c for c, _ in _spec})
print(f"  {'class':22s}{'train':>16s}{'valid':>14s}{'test':>14s}")
_thin_cls = []
for _c in sorted(_cls_all,
                 key=lambda c: -sum(len(_spec.get((c, s), ())) for s in ("train","valid","test"))):
    _cells = []
    for _sp in ("train", "valid", "test"):
        _ns = len(_spec.get((_c, _sp), ()))
        _nb = int(((_DF.split == _sp) & (_DF["class"] == _c)).sum())
        _cells.append(f"{_ns:>5d} ({_nb:,})".rjust(14 if _sp != "train" else 16))
    print(f"  {_c:22s}" + "".join(_cells))
    if len(_spec.get((_c, "test"), ())) < 3 or len(_spec.get((_c, "train"), ())) < 14:
        _thin_cls.append(_c)
if _thin_cls:
    print(f"\n  Fewer than 14 training or 3 test individuals: {_thin_cls}")
    print("  These are specimen-limited, not model-limited. Whatever AP they score is a")
    print("  statement about how many animals BIOSCAN holds, not about the detector —")
    print("  quote the specimen count next to the AP whenever you report them.")

print("\nVERIFIED")
for _l in _vok: print("  OK  " + _l)
print(f"\n{COCO_DATASET_DIR} is ready. Next: mix_real_and_synthetic.ipynb.")
